# PDF → editable Word (native equations)
Every question, option and formula comes out as editable text or a native Word equation; only diagrams are pictures.

**How to use**
1. *Runtime → Change runtime type → **T4 GPU*** → Save.
2. *Runtime → **Run all***.
3. A **Choose Files** button appears right below the first cell: pick your PDF(s).
4. Wait, and **keep this tab open**: free Colab may stop a notebook whose tab is closed. The Word file **downloads by itself** at the end (several PDFs come as one `.zip`). If the browser asks, allow the download.
5. Read the last lines under the *Download* cell. `CHECK against the PDF -> page …` names every page where something could not be read cleanly; compare those pages with the PDF.

Nothing else to click. A page takes about 1–2 minutes on the T4. If it stops halfway while the runtime is still connected, run the *Convert* cell again: pages already read are kept.

In [ ]:
import sys, torch
ALLOW_CPU = False   # True = convert without a GPU anyway: far slower, and much weaker on scanned pages
assert sys.version_info[:2] in ((3, 11), (3, 12), (3, 13)), "Use a Python 3.11-3.13 runtime"
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif not ALLOW_CPU:
    raise RuntimeError("No GPU on this runtime. Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all. "
                       "Free Colab sometimes has no GPU to give: try again later, or set ALLOW_CPU = True above "
                       "to convert anyway (far slower, much weaker on scanned pages).")
else:
    print("!" * 70 + "\n  NO GPU (ALLOW_CPU is on): the conversion is far slower and much weaker on scanned pages.\n" + "!" * 70)

from google.colab import files
print("\nChoose the PDF(s) to convert:")
up = files.upload()
PDFS = [n for n in up if n.lower().endswith(".pdf")]
skipped = [n for n in up if n not in PDFS]
if skipped:
    print("Not a PDF, skipped:", ", ".join(skipped))
if not PDFS:
    raise RuntimeError("No PDF chosen. Run this cell again (or Runtime > Run all) and pick a .pdf file.")
print("Will convert:", ", ".join(PDFS))

## Install (takes a few minutes)

In [ ]:
%%bash
set -e
pip -q install --retries 3 --timeout 60 paddlepaddle==3.2.2 "paddleocr[doc-parser]==3.3.2"
pip -q install --retries 3 --timeout 60 onnxruntime onnx ultralytics huggingface_hub tokenizers pymupdf python-docx pypandoc_binary lxml pillow transformers==5.22.2 accelerate==1.15.0
# Colab preinstalls langchain 1.x; paddlex imports langchain.docstore (removed in 1.x) whenever langchain is present
pip -q uninstall -y langchain langchain-community || true


## Converter script

In [ ]:
%%writefile pdf2word_hybrid.py
#!/usr/bin/env python3
"""
pdf2word_hybrid.py  --  STEM PDF  ->  editable Word (.docx) with native OMML equations.

Pipeline (per page, every stage cached as JSON "IR" so a bad page can be hand-edited):

  PyMuPDF raster @300dpi -> PaddleOCR PP-DocLayout_plus-L layout boxes (CPU, paddlepaddle 3.2.2)
  engine "vlm" (CUDA GPU, e.g. Colab T4; default when available):
    -> coverage pass: ink the layout model left unboxed (option labels, '8.', option formulas) is added
    -> PaddleOCR-VL-1.5 (0.9B, fp16) reads each region: text with inline/display LaTeX in one pass
  engine "hybrid" (CPU fallback):
    -> Pix2Text MFD-1.5 formula boxes + PaddleOCR PP-OCRv5 text on a page with formulas whited-out
    -> Pix2Text MFR-1.5 LaTeX (ONNX via onnxruntime; Paddle PP-FormulaNet as fallback)
  both:
    -> running heads / page numbers dropped, figures cropped (the only pictures in the output)
    -> reading order (column-aware), then questions/options rebuilt from the printed markers '12.' '(A)'
    -> python-docx + pandoc(OMML): native Word equations; anything pandoc can't convert is written as text
    -> output.docx

WHY NOT the `pix2text` PACKAGE?  It hard-imports cnstd/cnocr -> rapidocr (old pin, no py3.13).
We only need two of its *models* (MFD, MFR). Both are published on Hugging Face as plain ONNX files
(no PyTorch weights), so we run them with onnxruntime and never touch the dependency chain.

Install (Python 3.11-3.13; every pin below has cp313 wheels on PyPI, none come from Baidu's CDN)
  pip install paddlepaddle==3.2.2 "paddleocr[doc-parser]==3.3.2" onnxruntime onnx ultralytics \
              huggingface_hub tokenizers pymupdf python-docx pypandoc_binary lxml pillow
  (paddlepaddle 3.3.0 crashes CPU inference in PIR->oneDNN. On the pinned 3.2.x oneDNN is safe and ~5x faster, so it is
   enabled there only; any other paddle version runs with enable_mkldnn=False.)

Usage
  python pdf2word_hybrid.py in.pdf -o output.docx --work work               # --engine vlm|hybrid to force one
  python pdf2word_hybrid.py in.pdf -o output.docx --work work --rebuild     # after hand-editing work/ir/*.json
  python pdf2word_hybrid.py --check-env
"""
from __future__ import annotations

import functools, argparse, copy, difflib, json, math, os, re, statistics, sys, time, traceback, zipfile
from html.parser import HTMLParser
from pathlib import Path

print = functools.partial(print, flush=True)   # Colab buffers piped stdout otherwise
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(errors="replace")    # Windows cp1252 consoles can't print ρ / ≥ / ✔

# --------------------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------------------
CFG = dict(
    dpi=300,
    ocr_profile="fast",            # "fast" = mobile det/rec (~5 s/page CPU); "accurate" = server (~70-100 s/page CPU)
    layout_model=None,             # None -> PPStructureV3 default (PP-DocLayout_plus-L)
    mfd_imgsz=1280, mfd_conf=0.25,
    mfr_models=["breezedeus/pix2text-mfr-1.5", "breezedeus/pix2text-mfr"],
    mfd_repos=["breezedeus/pix2text-mfd-1.5", "breezedeus/pix2text-mfd"],
    paddle_formula_model="PP-FormulaNet_plus-M",
    formula_pad_x=10, formula_pad_y=6,
    lum_thresh=245,                # ink = luminance < 245  => keeps light-grey grid/axes
    fig_expand=60, fig_join=21, fig_pad=8,
    subject="physics",
    batch=16,
    tables=False,                  # table models are slow to load; MCQ banks rarely need them
    mfd_weights=None,              # local path / URL override for the formula-detector weights (.pt or .onnx)
    engine="auto",                 # "vlm" (PaddleOCR-VL-1.5, needs CUDA) | "hybrid" (CPU OCR+MFD/MFR) | "auto"
    vlm_model="PaddlePaddle/PaddleOCR-VL-1.5",
)

DROP_LABELS = {"header", "footer", "page_number", "number", "footer_image", "header_image"}
FIG_LABELS = {"image", "figure", "chart"}
TABLE_LABELS = {"table"}
FORMULA_LABELS = {"formula", "display_formula"}
EQNUM_LABELS = {"formula_number"}

MONTH = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?"
TAG_ITEM = rf"{MONTH}\s*\d{{2,4}}"
TAG_RE = re.compile(rf"\[\s*(?:{TAG_ITEM}(?:\s*[,;/&]\s*(?:{TAG_ITEM}|\d{{2,4}}))*|\d+\s*marks?)\s*(?:\]|$)", re.I)
GARBAGE_RES = [re.compile(r"(?:\d\s?\.\s?){4,}\d?")]          # "1.1.1.1.1.1"
Q_START_RE = re.compile(r"^\s*\(?(\d{1,3})\s*[.)](?!\d)\s*")      # "12." but not "2.5 kg"
# '6.8 Sonometer', '1.3 Applications of Uniform Circular Motion'; a 4+ letter word so '2.5 Hz' / '1.5 N' stay text
SECTION_RE = re.compile(r"\s*\d{1,2}\.\d{1,2}\s+[A-Z][a-z]{3,}(?:\s+[A-Za-z,&()\-₀-₉⁰-⁹⁺⁻]+)*\s*")   # '(KMnO₄ and K₂Cr₂O₇)'

# Heuristic symbol repairs (pattern, replacement, tag). Applied to LaTeX only. Add your own with --rules.
SUBJECT_RULES = {
    "physics": [
        # rho read as upright p; not pH/pK/pOH/ppm, nor a pico- unit after a number ('5\mathrm{pF}', '2\,\mathrm{pm}')
        (r"\\mathrm\{p(?!H\b|K\b|OH\b|pm\b)(?!(?<=[\d~,]\\mathrm\{p)[FCANWJVsm]\})([A-Za-z]*)\}", r"\\rho \1", "rho"),
        (r"(\\(?:sin|cos|tan|cot|sec|csc))\s*0(?![\d.^])", r"\1\\theta", "theta"),   # theta read as 0; 'sin 0^\circ' stays
        (r"(?<=\d)\s*[xX]\s*(?=10\b)", r"\\times ", "times"),
    ],
    "chemistry": [(r"(?<=\d)\s*[xX]\s*(?=10\b)", r"\\times ", "times")],
    "math": [(r"(\\(?:sin|cos|tan|cot|sec|csc))\s*0(?![\d.^])", r"\1\\theta", "theta")],
}

OMML_NS = "http://schemas.openxmlformats.org/officeDocument/2006/math"
W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
NS = {"w": W_NS, "m": OMML_NS}


# --------------------------------------------------------------------------------------
# Environment
# --------------------------------------------------------------------------------------
def configure_env(cache_dir: str | None = None):
    """Avoid the Baidu BOS CDN (timeouts) and the oneDNN/PIR crash; allow persistent model cache."""
    e = os.environ
    e.setdefault("PADDLE_PDX_MODEL_SOURCE", "huggingface")
    e.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
    e.setdefault("FLAGS_use_mkldnn", "0")
    e.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")
    e.setdefault("HF_HUB_ETAG_TIMEOUT", "15")
    e.setdefault("TOKENIZERS_PARALLELISM", "false")
    e.setdefault("YOLO_AUTOINSTALL", "false")      # ultralytics would otherwise pip-install mid-run
    try:
        import torch  # noqa: F401  (ultralytics needs it; on Windows it must load before paddle: WinError 127)
    except ImportError:
        pass
    if cache_dir:
        Path(cache_dir).mkdir(parents=True, exist_ok=True)
        e["HF_HOME"] = str(Path(cache_dir) / "hf")
        e["PADDLE_PDX_CACHE_HOME"] = str(Path(cache_dir) / "paddlex")


def check_env() -> bool:
    ok = True
    print(f"python {sys.version.split()[0]}")
    for mod, hint in [("pymupdf", "pymupdf"), ("cv2", "opencv-contrib-python (comes with paddlex)"), ("numpy", "numpy"),
                      ("PIL", "pillow"), ("docx", "python-docx"), ("lxml", "lxml"),
                      ("pypandoc", "pypandoc_binary"), ("paddle", "paddlepaddle==3.2.2"),
                      ("paddleocr", "paddleocr[doc-parser]==3.3.2"), ("onnxruntime", "onnxruntime"),
                      ("onnx", "onnx"), ("tokenizers", "tokenizers"), ("ultralytics", "ultralytics"),
                      ("huggingface_hub", "huggingface_hub")]:
        try:
            m = __import__(mod)
            print(f"  ok   {mod:16s} {getattr(m, '__version__', '')}")
        except Exception as ex:  # noqa
            ok = False
            print(f"  FAIL {mod:16s} pip install {hint}   ({type(ex).__name__}: {str(ex)[:80]})")
    if sys.version_info[:2] not in ((3, 11), (3, 12), (3, 13)):
        ok = False
        print("  FAIL python: use 3.11-3.13 (onnxruntime needs >=3.11; paddlepaddle 3.2.2 has no wheels past 3.13)")
    try:
        import pypandoc
        print("  pandoc:", pypandoc.get_pandoc_version())
    except Exception as ex:
        ok = False
        print("  FAIL pandoc   pip install pypandoc_binary  ", ex)
    print("  engine:", "vlm (GPU, best quality)" if gpu_vlm_available() else
          "hybrid (CPU). For the GPU engine: CUDA torch + pip install transformers==5.22.2 accelerate==1.15.0")
    return ok


def construct(cls, **kw):
    """Instantiate a PaddleOCR/PaddleX class, dropping kwargs this version doesn't know (API drift)."""
    kw = dict(kw)
    for _ in range(12):
        try:
            return cls(**kw)
        except TypeError as ex:
            m = re.search(r"unexpected keyword argument '(\w+)'", str(ex))
            if not m or m.group(1) not in kw:
                raise
            print(f"   [compat] {cls.__name__}: dropping unsupported arg {m.group(1)}")
            kw.pop(m.group(1))
        except ValueError as ex:
            m = re.search(r"[Uu]nknown argument[s]?:? \[?'?(\w+)", str(ex))
            if not m or m.group(1) not in kw:
                raise
            print(f"   [compat] {cls.__name__}: dropping unsupported arg {m.group(1)}")
            kw.pop(m.group(1))
    return cls(**kw)


# --------------------------------------------------------------------------------------
# Geometry
# --------------------------------------------------------------------------------------
def area(b): return max(0, b[2] - b[0]) * max(0, b[3] - b[1])
def inter(a, b): return area([max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])])
def iou(a, b):
    i = inter(a, b); u = area(a) + area(b) - i
    return i / u if u else 0.0
def center(b): return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)
def contains_pt(b, p, tol=0): return b[0] - tol <= p[0] <= b[2] + tol and b[1] - tol <= p[1] <= b[3] + tol
def expand(b, dx, dy): return [b[0] - dx, b[1] - dy, b[2] + dx, b[3] + dy]
def union(a, b): return [min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3])]
def vover(a, b):
    o = min(a[3], b[3]) - max(a[1], b[1])
    return max(0, o) / max(1, min(a[3] - a[1], b[3] - b[1]))


# --------------------------------------------------------------------------------------
# LaTeX / text hygiene
# --------------------------------------------------------------------------------------
def clean_latex(s: str) -> str:
    """MFR emits space-separated tokens ('\\frac { a } { b }'). Rejoin, keeping the space only
    where a control word is followed by a letter."""
    out = ""
    for t in s.strip().split():
        bs = len(out) - len(out.rstrip("\\"))       # odd = a lone '\': the control space 'a \ b', not a '\\' row break
        if out and (bs % 2 or re.search(r"\\[A-Za-z]+$", out) and re.match(r"[A-Za-z]", t)):
            out += " "
        out += t
    return re.sub(r"\\operatorname\{(sin|cos|tan|cot|sec|csc|log|ln|exp)\}", r"\\\1", out)   # MFR-1.5 habit


def balance_braces(s: str) -> str:
    depth, out, esc = 0, [], False
    for ch in s:
        if esc:                     # \{ \} are literal braces, but in '\\{' the brace opens a group (matrix rows)
            esc = False
        elif ch == "\\":
            esc = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            if depth == 0:
                continue
            depth -= 1
        out.append(ch)
    return "".join(out) + "}" * depth


GREEK = {"alpha": "α", "beta": "β", "gamma": "γ", "delta": "δ", "epsilon": "ε", "theta": "θ", "lambda": "λ", "mu": "μ",
         "pi": "π", "rho": "ρ", "sigma": "σ", "tau": "τ", "phi": "φ", "omega": "ω", "Delta": "Δ", "Omega": "Ω",
         "times": "×", "cdot": "·", "pm": "±", "degree": "°", "circ": "°"}
# latex_text only: without these a formula pandoc rejects lost its relations ('a \leq b' -> 'a b')
SYMBOLS = dict(GREEK, nu="ν", eta="η", kappa="κ", xi="ξ", psi="ψ", chi="χ", zeta="ζ", varepsilon="ε", varphi="φ",
               vartheta="θ", Gamma="Γ", Theta="Θ", Lambda="Λ", Pi="Π", Sigma="Σ", Phi="Φ", Psi="Ψ", hbar="ħ", ell="ℓ",
               leq="≤", le="≤", leqslant="≤", geq="≥", ge="≥", geqslant="≥", neq="≠", ne="≠", approx="≈", simeq="≃",
               cong="≅", equiv="≡", sim="∼", propto="∝", ll="≪", gg="≫", infty="∞", partial="∂", nabla="∇",
               rightarrow="→", to="→", longrightarrow="→", leftarrow="←", leftrightarrow="↔", Rightarrow="⇒",
               Leftarrow="⇐", Leftrightarrow="⇔", rightleftharpoons="⇌", uparrow="↑", downarrow="↓", int="∫", sum="∑",
               prod="∏", angle="∠", perp="⊥", parallel="∥", div="÷", mp="∓", cup="∪", cap="∩", subset="⊂",
               subseteq="⊆", notin="∉", emptyset="∅", forall="∀", exists="∃", therefore="∴", because="∵",
               ldots="…", cdots="⋯", dots="…", neg="¬", wedge="∧", vee="∨", oplus="⊕", otimes="⊗", prime="′", mid="|")
SYMBOLS["in"] = "∈"
SYMBOL_RE = re.compile(r"\\(" + "|".join(sorted(SYMBOLS, key=len, reverse=True)) + r")(?![A-Za-z])")
SUP, SUB = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻"), str.maketrans("0123456789-", "₀₁₂₃₄₅₆₇₈₉₋")


def latex_text(s: str) -> str:
    """Readable plain text for LaTeX that pandoc can't turn into a Word equation: editable text beats a picture."""
    s = re.sub(r"\\(?:mathrm|text|textrm|mathbf|mathit|mathsf|mathtt|textsc|operatorname|boldsymbol)(?![A-Za-z])\s*", "", s)
    # matrices/arrays: '[2 -5; 3 -1]', not 'bmatrix2&-5 3&-1bmatrix'
    s = re.sub(r"\\begin\{array\}\s*\{[^{}]*\}", " ", s)
    s = re.sub(r"\\(begin|end)\{([pbBvV]?)matrix\}",
               lambda m: dict(p="()", b="[]", B="{}", v="||", V="‖‖").get(m.group(2), "  ")[m.group(1) == "end"], s)
    s = re.sub(r"\\(?:begin|end)\{[A-Za-z*]+\}", " ", s).replace("\\\\", "; ").replace("&", " ")
    s = re.sub(r"\^\s*\{?\s*\\prime\s*\}?", "′", s)                 # Avogadro^{\prime}s -> Avogadro′s
    s = SYMBOL_RE.sub(lambda m: SYMBOLS[m.group(1)], s)
    s = re.sub(r"\^(?:\{(-?[0-9]+)\}|([0-9]))", lambda m: (m.group(1) or m.group(2)).translate(SUP), s)   # x^{12}, x^{-1}
    s = re.sub(r"_(?:\{(-?[0-9]+)\}|([0-9]))", lambda m: (m.group(1) or m.group(2)).translate(SUB), s)
    for _ in range(8):                              # innermost first: fractions, roots, then plain {groups}
        s0 = s
        s = re.sub(r"\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}", r"(\1)/(\2)", s)
        s = re.sub(r"\\sqrt\s*\{([^{}]*)\}", r"√(\1)", s)
        s = re.sub(r"(?<![A-Za-z}])\{([^{}]*)\}", r"\1", s)
        if s == s0:
            break
    s = re.sub(r"\\(?:(?:left|right|q?quad)(?![A-Za-z])|[,;:! ])", " ", s)
    s = re.sub(r"\\((?:arc)?(?:sin|cos|tan)h?|cot|sec|csc|log|ln|lim|exp|det|max|min)(?![A-Za-z])", r"\1 ", s)   # names stay
    s = re.sub(r"\\[A-Za-z]+", "", s).replace("\\", "")
    return re.sub(r"\s+", " ", re.sub(r"[{}~]", lambda m: " " if m.group() == "~" else "", s)).strip()


def textlike(lx: str) -> bool:
    """MFD sometimes boxes plain text ('(A) 3 m/s (B)', '7. If L'); its LaTeX then carries option labels,
    question numbers or multi-word \\mathrm/\\text runs. Such runs belong in the text, not in an equation."""
    # checked on the raw LaTeX: linearised, \frac{v}{D} would read '(v)/(D)' and look like an option label
    lx = re.sub(r"(?<!\\)\\(?:q?quad(?![A-Za-z])|[,;: ])", " ", lx)    # '\quad(D)': the label follows a space
    return bool(re.search(r"(?<![A-Za-z])\(\s*(?:\\(?:mathrm|text)\s*\{)?[A-D]\}?\s*\)(?!\s*[\^_'])|^\s*\d{1,3}\.\s", lx)   # not P(A), (A)^{-1}
                or re.search(r"\\(?:mathrm|text)\s*\{[^{}]*[A-Za-z]{2,}[ ~]+[A-Za-z]", lx)
                or re.fullmatch(r"\s*(?:\\(?:text|textrm|mathrm)\s*\{)?\s*[A-Za-z]{4,}(?:\s+[A-Za-z]{2,})*\s*\}?\s*", lx))   # \text{linear}


def _depths(s: str) -> list:
    """Nesting depth before each char (and after the last): braces, \\begin..\\end, \\left..\\right."""
    out, d, i = [], 0, 0
    while i < len(s):
        if s[i] == "\\":
            m = re.match(r"\\([A-Za-z]+|.?)", s[i:], re.S); name, n = m.group(1), len(m.group(0))
            d -= name in ("end", "right")
            out += [d] * n
            d += name in ("begin", "left"); i += n; continue
        d -= s[i] == "}"
        out.append(d)
        d += s[i] == "{"; i += 1
    return out + [d]


def _top_split(s: str, sep: str) -> list:
    dep, parts, pos, i = _depths(s), [], 0, 0
    while i < len(s):
        if dep[i] == 0 and s.startswith(sep, i):
            parts.append(s[pos:i]); i += len(sep); pos = i
        elif s[i] == "\\":
            i += 2                                  # never split inside '\\' or '\&'
        else:
            i += 1
    return parts + [s[pos:]]


def _unbrace(s: str) -> str:
    s = re.sub(r"(?<!\\)\{\s*\}", "", s).strip()
    while s.startswith("{") and s.endswith("}") and len(s) > 1 and 0 not in _depths(s)[1:-2]:
        s = s[1:-1].strip()                         # '{{5.}}' -> '5.' (the braces wrap the whole thing)
    return s


TEXT_ENV_RE = re.compile(r"\s*\\begin\{(array|aligned|align\*?|gathered|split|matrix)\}(\s*\{[a-z| ]*\})?(.*)"
                         r"\\end\{\1\}[\s,.;}]*", re.S)


def unwrap_env(lx: str) -> str:
    """The VLM wraps a whole printed line in one array/aligned: '\\begin{aligned}6.\\quad&If A=...\\end{aligned}'.
    -> its cells joined by spaces, rows by newlines. Anything else is returned unchanged."""
    m = TEXT_ENV_RE.fullmatch(lx)
    if not m:
        return lx
    body = (m.group(2) or "") + m.group(3) if m.group(1) != "array" else m.group(3)   # only array has a column spec
    dep = _depths(body)
    if min(dep) < 0 or dep[-1]:                     # '\begin{array}..\end{array} x \begin{array}..': two envs, not one
        return lx
    rows = [" ".join(c for c in map(_unbrace, _top_split(row, "&")) if c) for row in _top_split(body, "\\\\")]
    return "\n".join(r for r in rows if r)


# a label follows a space, a bracket or the end of a matrix ('\end{bmatrix}(D)\frac{1}{13}...': two options read as one
# formula); after any other '}' it is an argument ('\mathrm{adj}(A)', 'x^{2}(B)')
TEXTBIT_RE = re.compile(r"(?:(?<![A-Za-z0-9}])|(?<=\\end\{[pbBvV]matrix\})|(?<=\\end\{matrix\})|(?<=\\end\{array\}))"
                        r"\(\s*(?:\\(?:mathrm|text)\s*\{\s*)?(?P<lab>[A-D])\s*\}?\s*\)(?!\s*[\^_'])"   # (A), (\mathrm{B})
                        r"|(?:^|(?<=\n))[ \t]*\{*\s*(?P<qn>\d{1,3})\s*\.(?!\d)\s*\}*"                    # '6.' / '{{5.}}' at a line start
                        r"|\\(?:text|textrm|mathrm)\s*\{(?P<cmd>[A-Za-z ~.,:;'?-]+)\}")                  # \mathrm{I f}, \text{verify that}
# a bare word needs a vowel and no digit/letter/backslash before it: 'If', 'verify that' are text; '2xy', 'dx' are maths
# ... but an operator name is maths: 'A(adj A)', '|adj A|', 'det(A)', 'sin x' must not be cut out of their equation
OPS = r"(?:adj|det|diag|lim|log|exp|sin|cos|tan|cot|sec|csc)(?![A-Za-z])"
WORDS_RE = re.compile(rf"(?<![\\A-Za-z0-9(\[|])(?!{OPS})(?P<w>(?=[A-Za-z ~]*[aeiouAEIOU])[A-Za-z]{{2,}}"
                      rf"(?:[ ~]+(?!{OPS})[A-Za-z]{{2,}})*)(?![A-Za-z])"
                      r"|(?P<nl>\n)")


def split_textlike(lx: str) -> list:
    """A 'formula' that is really a printed line (option labels, a question number, words around matrices) ->
    text runs for the labels/words and math runs for the formulas between them. Flattening the whole thing to text
    turned matrix options into 'bmatrix1&5 3&-2bmatrix' and hid '(B)'/'6.' from the question structuring."""
    lx = re.sub(r"\\(?:text|textrm|mathrm)\s*\{\s*(\(\s*[A-D]\s*\))\s*\}", r" \1 ", lx)     # '...\end{bmatrix} \text{(D)} \frac..'
    s = re.sub(r"(?<!\\)\\(?:q?quad(?![A-Za-z])|[,;: ])", " ", unwrap_env(lx))
    dep = _depths(s)
    spans = [m for m in TEXTBIT_RE.finditer(s) if dep[m.start()] == 0 == dep[m.end()]
             and (m.group("cmd") is None or sum(c.isalpha() for c in m.group("cmd")) >= 2
                  and not re.fullmatch(OPS, m.group("cmd").strip()))]        # '\mathrm{adj}(A)': an operator, maths
    hide = {i for m in spans for i in range(m.start(), m.end())}
    masked = "".join("\0" if dep[i] or i in hide else ch for i, ch in enumerate(s))
    spans = sorted(spans + list(WORDS_RE.finditer(masked)), key=lambda m: m.start())
    runs, pos = [], 0
    for m in spans + [None]:
        piece = _unbrace(re.sub(r"^[\s~&]+|[\s~&]+$", "", s[pos:m.start() if m else len(s)]))
        if re.search(r"[^\W_]|\\", piece):
            runs.append(dict(t="math", v=sanitize_for_pandoc(piece), kind="inline", src="vlm", flags=[]))
        elif piece:
            runs.append({"t": "text", "v": f" {piece} "})
        if m is None:
            break
        g = m.groupdict()
        if g.get("lab"):
            t = f" ({g['lab']}) "
        elif g.get("qn"):
            t = f"\n{g['qn']}. "
        elif g.get("cmd") is not None:
            t = latex_text(m.group())
            t = f" {t.replace(' ', '') if re.fullmatch(r'(?:[A-Za-z] )+[A-Za-z]', t) else t} "   # '\mathrm{I f}' -> 'If'
        else:
            t = "\n" if g.get("nl") else f" {g['w'].replace('~', ' ')} "
        runs.append({"t": "text", "v": t}); pos = m.end()
    return runs


# --------------------------------------------------------------------------------------
# Chemistry: a formula is text with real subscripts, a reaction an equation in upright type
# --------------------------------------------------------------------------------------
ELEMENTS = set("H He Li Be B C N O F Ne Na Mg Al Si P S Cl Ar K Ca Sc Ti V Cr Mn Fe Co Ni Cu Zn Ga Ge As Se Br Kr Rb Sr Y "
               "Zr Nb Mo Tc Ru Rh Pd Ag Cd In Sn Sb Te I Xe Cs Ba La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu Hf Ta W Re Os "
               "Ir Pt Au Hg Tl Pb Bi Po At Rn Fr Ra Ac Th Pa U Np Pu Am Cm Bk Cf Es Fm Md No Lr Rf Db Sg Bh Hs Mt Ds Rg Cn Nh "
               "Fl Mc Lv Ts Og".split()) | {"R", "X", "Ph", "Me", "Et"}          # + the generic groups of organic formulas
SUB_TR, SUP_TR = str.maketrans("0123456789n", "₀₁₂₃₄₅₆₇₈₉ₙ"), str.maketrans("0123456789+-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻")
CHEM_TOK = re.compile(r"(?P<el>[A-Z][a-z]?)|_\{?(?P<sub>\d+|n)\}?|\^\{?(?P<chg>\d*[+-]|[+-]\d*|⊕|⊖)\}?|(?P<num>\d+)"
                      r"|(?P<state>_?\{?\((?:aq|s|l|g)\)\}?)"                      # state: Fe^{2+}(aq), H_{2}O_{(l)}
                      r"|(?P<br>[()\[\]])|(?P<bond>[-=≡])|(?P<dot>·)|(?P<plus>\+)|(?P<prime>')")
# a lone element with a subscript is a formula only as its molecule (H2, O3, P4, S8): P2, V2, I1 are variables
MOLECULES = {"H": {"2"}, "N": {"2"}, "O": {"2", "3"}, "F": {"2"}, "Cl": {"2"}, "Br": {"2"}, "I": {"2"}, "P": {"4"}, "S": {"8"}}
ARROW_RE = re.compile(r"\\(?:x?(?:long)?(?:right|left|leftright)arrow|rightleftharpoons|leftrightharpoons|to)(?![A-Za-z])")


def _unwrap_text_cmds(s):
    """'\\mathrm{C H_{3}-C H_{2}}' -> 'C H_{3}-C H_{2}' (balanced braces)."""
    while m := re.search(r"\\(?:mathrm|text|textrm|rm|mathbf|textbf)\s*(?=\{)", s):
        body, end = _brace_arg(s, m.end())
        s = s[:m.start()] + body + s[end:]
    return s


def chem_text(lx, loose=False):
    """'H_{2}SO_{4}', 'HSO_{4}^{-}', '\\mathrm{C H_{3}-C H_{2}-Br}', '2NaBr' -> 'H₂SO₄', 'HSO₄⁻', 'CH₃–CH₂–Br', '2NaBr':
    a chemical formula as text with real subscripts, upright as a chemist types it (a Word equation sets every
    letter in italic, as variables). None for anything that is not purely a formula: 'K_{sp}', 'S_{N}2', 'BA_{2}',
    'P_{1}V_{1}=P_{2}V_{2}', 'x^{2}', a lone 'H_{2}' or 'V_{2}' (too likely a variable).
    loose (a chemistry document): also 'CaO', 'CO', 'NaCl' without marks, the molecules 'H_{2}', 'P_{4}', and a
    state 'Fe^{2+}(aq)'; never 'PV' or 'P_{2}' (gas-law variables)."""
    s = _unwrap_text_cmds(lx)
    s = re.sub(r"\\(?:cdot|bullet)(?![A-Za-z])", "·", s).replace("\\equiv", "≡").replace("\\oplus", "⊕").replace("\\ominus", "⊖")
    s = re.sub(r"[−–—]|\\minus(?![A-Za-z])", "-", s)
    s = re.sub(r"\\[,;: ]|~|\{\}", "", s)
    s = re.sub(r"\s*\+\s*(?=\d*[A-Z])", " + ", s)                        # '+' between species, not a charge
    s = re.sub(r"(?<! )(?<!\+) (?! ?\+)", "", s.strip())                # the reader's 'C H_{3}': spaces mean nothing
    s = re.sub(r"\{([A-Za-z]+)\}", r"\1", s)
    organic = bool(re.search(r"C(?![a-z])|R|Ph|Me|Et", s))     # carbon (not Cl, Ca, Cu) or an organic group
    out, pos, els, marks, charged, two_letter_sub, prev = [], 0, 0, 0, False, False, ""
    syms, subs, state = [], [], False
    for m in CHEM_TOK.finditer(s):
        if s[pos:m.start()].strip():
            return None                             # anything that is not formula syntax: a variable, a command
        out.append(s[pos:m.start()]); pos = m.end()
        k, v = m.lastgroup, m.group()
        if k == "el":
            if v not in ELEMENTS:
                return None                         # 'A' in 'BA_{2}', 'M', ...
            els += 1; out.append(v); syms.append(v)
        elif k == "state":
            if not loose or prev not in ("el", "sub", "chg", "br_close"):
                return None
            state = True; out.append(re.sub(r"[_{}]", "", v))
        elif k == "sub" or (k == "num" and prev in ("el", "br_close")):    # 'H_{2}O', or mhchem-style 'H2O'
            if (m.group("sub") or v) in ("0", "1"):
                return None                         # no formula is written with a subscript 1 or 0: 'I_{1}', 'C_{1}V_{1}'
            marks += 1; out.append((m.group("sub") or v).translate(SUB_TR)); subs.append(m.group("sub") or v)
            two_letter_sub |= prev == "el" and len(out[-3]) == 2 if len(out) >= 3 else False
        elif k == "num":                            # a coefficient: at the start, after '·', '+' or '('
            if m.start() and not re.search(r"[·+(\s]$", s[:m.start()]):
                return None
            marks += 1; out.append(v)
        elif k == "chg":
            marks += 1; charged = True; c = m.group("chg")
            out.append(c if c in ("⊕", "⊖") else (c[1:] + c[0] if c[0] in "+-" else c).translate(SUP_TR))
        elif k == "bond":
            if not organic:
                return None                         # 'P_{1}V_{1}=P_{2}V_{2}' is physics: bonds are drawn in organic formulas
            marks += 1; out.append({"-": "–"}.get(v, v))
        else:
            out.append(v)
        prev = "br_close" if v in ")]" else k
    if s[pos:].strip() or not els:
        return None
    if not (els >= 3 or (els >= 2 and marks) or (els == 1 and (charged or two_letter_sub))):
        molecule = els == 1 and len(subs) == 1 and subs[0] in MOLECULES.get(syms[0], set())
        if not (loose and ((els >= 2 and sorted(syms) != ["P", "V"]) or molecule or (els == 1 and state))):
            return None                             # 'HCl', a lone 'H_{2}': left as they came outside chemistry
    return re.sub(r"\s+", " ", "".join(out)).strip()


def chem_reaction(lx):
    """A reaction ('CH_{3}CH_{2}Br \\xrightarrow[\\Delta]{alc.\\ KOH} ...') -> the same LaTeX with every letter upright
    (\\mathrm), so Word sets formulas and conditions as a chemist would; None when it is not one (an arrow with no
    chemical formula, 'x \\to 0', is mathematics)."""
    if not ARROW_RE.search(lx):
        return None
    flat = _unwrap_text_cmds(lx)
    if not any(chem_text(f) for f in re.findall(r"\d*(?:[A-Z][a-z]?(?:_\{?\d+\}?)?){2,}(?:\^\{?\d*[+-]\}?)?", flat)):
        return None
    s = re.sub(r"\\(?:text|textrm)\s*\{([^{}]*)\}", lambda m: r"\mathrm{" + m.group(1).strip().replace(" ", r"\ ") + "}", lx)
    return re.sub(r"\\mathrm\s*\{[^{}]*\}|\\(?:begin|end)\s*\{[^{}]*\}|\\[A-Za-z]+|([A-Za-z]+)",
                  lambda m: rf"\mathrm{{{m.group(1)}}}" if m.group(1) else m.group(), s)


def text_only_math(lx):
    """A 'formula' made of nothing but \\mathrm{...}/\\text{...} groups (a '[Given: Avogadro's number = ...]' line the
    reader wrapped as an array) -> its plain text, with the letters the reader spaced out ('G i v e n') joined again.
    None for real maths, and for a lone exam tag ('\\text{Mar 18}'), which finish_page turns into the question's tag."""
    body = re.sub(r"\\begin\s*\{array\}\s*\{[^{}]*\}|\\(?:begin|end)\s*\{[^{}]*\}|\\\\|&", " ", lx)   # rows: one line
    s = body
    while m := re.search(r"\\(?:mathrm|text|textrm)\s*(?=\{)", s):
        s = s[:m.start()] + " " + s[_brace_arg(s, m.end())[1]:]
    if s.strip(" {}.,;:") or not re.search(r"\\(?:mathrm|text|textrm)\b", lx):
        return None                                 # something outside the text groups: maths
    if set(re.findall(r"\\([A-Za-z]+)", body)) - {"mathrm", "text", "textrm", "prime", "times", "AA", "circ", "cdot", "quad", "qquad"}:
        return None                                 # an arrow, fraction ... inside them: a reaction, maths ('\mathrm{A\xrightarrow{}B}')
    out, pos = "", 0                                # inside \mathrm a plain space means nothing (as in LaTeX): the
    while m := re.search(r"\\mathrm\s*(?=\{)", body[pos:]):     # reader spaces letters ('G i v e n') and marks
        arg, end = _brace_arg(body, pos + m.end())               # word gaps with '~'
        out += body[pos:pos + m.start()] + r"\mathrm{" + arg.replace(" ", "") + "}"; pos = end
    t = re.sub(r"\s+", " ", latex_text(out + body[pos:])).strip()
    return None if TAG_RE.fullmatch(t) or TAG_RE.fullmatch(f"[{t}]") or not t else t


def untex(s):
    """LaTeX the reader left in plain text ('fuming H_{2}SO_{4}', 'CaCl_{2} is 111 g mol^{-1}') -> real sub/superscripts.
    Only the braced forms after a letter, digit or bracket: plain text never holds them otherwise."""
    s = re.sub(r"\^\{?\\circ\}?", "°", s)
    s = re.sub(r"(?<=[A-Za-z0-9)\]])_\{(\d+)\}", lambda m: m.group(1).translate(SUB_TR), s)
    return re.sub(r"(?<=[A-Za-z0-9)\]])\^\{([+\-−]?\d+[+\-]?)\}", lambda m: m.group(1).replace("−", "-").translate(SUP_TR), s)


LONE_SCRIPT_RE = re.compile(r"\s*([_^])\s*\{?\s*([+\-−]?\d+[+\-]?|\\circ)\s*\}?\s*")
ECONF_TERM = r"([1-7])\s*(?:\\mathrm\s*\{\s*)?([spdf])\s*\}?\s*\^\s*\{?\s*(\d+(?:\s*[-−–]\s*\d+)?)\s*\}?"
ECONF_RE = re.compile(rf"\s*(?:\\?\[\s*(?:\\mathrm\s*\{{\s*)?(He|Ne|Ar|Kr|Xe|Rn)\s*\}}?\s*\\?\]\s*)?((?:{ECONF_TERM}\s*)+)")


def econfig_text(lx):
    """'[Ar] 3d^{1-10}4s^{2}' -> '[Ar] 3d¹⁻¹⁰ 4s²': an electron configuration, set upright like a chemist's."""
    m = ECONF_RE.fullmatch(lx)
    if not m:
        return None
    terms = []
    for n, o, e in re.findall(ECONF_TERM, m.group(2)):
        e = re.sub(r"\s+", "", e).replace("−", "-").replace("–", "-")    # (no backslash inside f-strings: Python 3.11)
        terms.append(n + o + e.translate(SUP_TR))
    return (f"[{m.group(1)}] " if m.group(1) else "") + " ".join(terms)


def is_chem_doc(blocks):
    """A chemistry document: 3+ formulas, in maths runs or already in text ('H₂SO₄', 'CO_{2}'). On the reference
    documents chemistry chapters count 6-18, Matrices and a physics chapter 0; Solid State (2: CO₂, SO₂) and Polymer
    (0, its formulas are drawn) stay out, so they keep the strict rules: a bar a physics chapter naming CO₂ won't clear."""
    n = 0
    for b in blocks:
        for r in b["runs"]:
            if r["t"] == "math" and r.get("v"):
                n += bool(chem_text(r["v"]))
            elif r["t"] == "text":
                n += sum(1 for f in re.findall(r"(?<![A-Za-z])(?:[A-Z][a-z]?(?:[₀-₉]+|_\{\d+\})?){2,}", r["v"])
                         if re.search(r"[₀-₉]|_\{", f) and all(e in ELEMENTS for e in re.findall(r"[A-Z][a-z]?", f)))
    return n >= 3


def chemify(runs, chem=False):
    """Formulas in maths runs -> text with subscripts; reactions -> upright equations; a sub/superscript the reader
    split off its word ('KMnO' + '_{4}', 'cm' + '^{-3}') -> joined as text. In a chemistry document (chem) also the
    unmarked formulas and electron configurations. Other maths is untouched."""
    out = []
    for r in runs:
        if r["t"] == "text" and ("_{" in r["v"] or "^{" in r["v"] or "^\\circ" in r["v"]):
            r = dict(r, v=untex(r["v"]))
        if r["t"] == "math" and r.get("v"):
            m = LONE_SCRIPT_RE.fullmatch(r["v"])
            if (m and out and out[-1]["t"] == "text" and re.search(r"[A-Za-z0-9)\]]$", out[-1]["v"])
                    and (m.group(1) == "^" or m.group(2).isdigit())):      # no subscript signs in Unicode text
                g = m.group(2).replace("−", "-")
                out[-1] = dict(out[-1], v=out[-1]["v"] + ("°" if g == "\\circ" else g.translate(SUB_TR if m.group(1) == "_" else SUP_TR)))
                continue
            t = chem_text(r["v"], loose=chem) or (econfig_text(r["v"]) if chem else None)
            if t:
                out.append({"t": "text", "v": t}); continue
            rx = chem_reaction(r["v"])
            if rx:
                r = dict(r, v=rx)
        out.append(r)
    return merge_text_runs(out)


def cell_runs(text, chem=False):
    return chemify(vlm_runs(text), chem)


def _brace_arg(s, i):
    """(content, index after the closing brace) of the {...} group opening at s[i]."""
    depth = 0
    for j in range(i, len(s)):
        depth += (s[j] == "{") - (s[j] == "}")
        if depth == 0:
            return s[i + 1:j], j + 1
    return s[i + 1:], len(s)


def ce_to_latex(c):
    """mhchem '\\ce{CH3CH2Cl ->[AgCN][alc. \\Delta] ?}', which pandoc can't read -> plain LaTeX."""
    label = lambda t: r"\mathrm{" + re.sub(r"(?<=[A-Za-z)])(\d+)", r"_{\1}", t.strip()).replace(" ", r"\ ") + "}"

    def arrow(m):
        sym, above, below = m.group(1), m.group(2), m.group(3)
        if above is None:
            return " " + {"->": r"\rightarrow", "<-": r"\leftarrow", "<=>": r"\rightleftharpoons", "<->": r"\leftrightarrow"}[sym] + " "
        if sym in ("->", "<-"):
            x = r"\xrightarrow" if sym == "->" else r"\xleftarrow"
            return f" {x}[{label(below)}]{{{label(above)}}} " if below else f" {x}{{{label(above)}}} "
        return rf" \overset{{{label(above)}}}{{\rightleftharpoons}} "
    parts = re.split(r"((?:<=>|<->|->|<-)(?:\[[^\]]*\])?(?:\[[^\]]*\])?)", c)
    for k, p in enumerate(parts):
        m = re.fullmatch(r"(<=>|<->|->|<-)(?:\[([^\]]*)\])?(?:\[([^\]]*)\])?", p)
        if m:
            parts[k] = arrow(m)
        else:                                       # species: digits after a symbol are subscripts, '^2-' a charge
            p = re.sub(r"(?<=[A-Za-z)\]])(\d+)", r"_{\1}", p)
            parts[k] = re.sub(r"\^(\d*[+-])", r"^{\1}", p)
    return "".join(parts).strip()


def sanitize_for_pandoc(s: str) -> str:
    while (i := s.find("\\ce{")) >= 0 or (i := s.find("\\ce {")) >= 0:
        body, end = _brace_arg(s, s.index("{", i))
        s = s[:i] + ce_to_latex(body) + s[end:]
    s = re.sub(r"\\AA(?![A-Za-z])|\\mathring\s*\{\s*A\s*\}", "Å", s)          # angstrom: pandoc rejects \AA
    s = re.sub(r"\\up(" + "|".join(k for k in GREEK if k.isalpha()) + r")(?![A-Za-z])", r"\\\1", s)   # upgreek \uprho
    s = re.sub(r"\\(?:displaystyle|textstyle|scriptstyle|limits|nolimits)\b", "", s)
    s = re.sub(r"\\(?:tfrac|dfrac)\b", r"\\frac", s)
    s = s.replace("\\!", "").replace("$", "")
    s = re.sub(r"\\tag\*?\{[^}]*\}", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"(?:\\\\\s*)+$", "", s)
    s = re.sub(r"[\^_]\s*\{\s*\}", "", s)                              # empty ^{} / _{} break pandoc
    if len(re.findall(r"\\left\b", s)) != len(re.findall(r"\\right\b", s)):
        s = re.sub(r"\\(?:left|right)\b\s*\.?", "", s)                  # unbalanced -> plain delimiters
    return balance_braces(s)


def apply_rules(latex: str, rules, flags: list) -> str:
    for pat, rep, tag in rules:
        new = re.sub(pat, rep, latex)
        if new != latex:
            flags.append(f"rule:{tag}")
            latex = new
    return latex


def clean_text(s: str, flags: list) -> str:
    for g in GARBAGE_RES:
        if g.search(s):
            s = g.sub(" ", s); flags.append("garbage_removed")
    # velocity(B) -> velocity (B): a glued option label. P(A), n(A), det(A) stay: a function of A, and the space
    # would turn it into an option candidate
    s = re.sub(r"([A-Za-z]*)(?<=[A-Za-z0-9.,;:)])\((?=[A-D]\))",
               lambda m: m.group(0) if len(m.group(1)) == 1 or re.fullmatch(OPS, m.group(1)) else m.group(1) + " (", s)
    s = re.sub(r"\(([A-D])\)(?=[A-Za-z0-9])", r"(\1) ", s)                 # (A)text -> (A) text
    s = re.sub(r"(?<=\d) [xX] (?=\d)", " × ", s)
    s = re.sub(r"[ \t]{2,}", " ", s)
    return s


def runs_text(runs) -> str:
    return "".join(r["v"] if r["t"] == "text" else f"${r.get('v') or '?'}$" for r in runs)


def merge_text_runs(runs):
    out = []
    for r in runs:
        if r["t"] == "text" and out and out[-1]["t"] == "text":
            out[-1]["v"] += r["v"]
        else:
            out.append(dict(r))
    return out


# --------------------------------------------------------------------------------------
# Tags ([Mar 10], [3 marks]) and equation numbers
# --------------------------------------------------------------------------------------
def extract_tags(runs):
    tags = []
    for r in runs:
        if r["t"] != "text":
            continue
        for m in TAG_RE.finditer(r["v"]):
            t = re.sub(r"\s+", " ", m.group(0).strip()).replace("[ ", "[").replace(" ]", "]")
            t += "" if t.endswith("]") else "]"
            t = re.sub(r"(?<=[A-Za-z.])(?=\d)", " ", t)              # '[Mar17]' (read as maths) -> '[Mar 17]'
            tags.append(t)                          # repeats kept: two questions in one region can share a tag
        r["v"] = TAG_RE.sub(" ", r["v"])
    return tags


# --------------------------------------------------------------------------------------
# Reading order: column-aware XY-cut
# --------------------------------------------------------------------------------------
def order_blocks(blocks):
    """Returns blocks ordered, each tagged b['mode'] (1 or 2 columns) and b['col']."""
    if not blocks:
        return []
    x1 = min(b["bbox"][0] for b in blocks); x2 = max(b["bbox"][2] for b in blocks)
    CW = max(1.0, x2 - x1)

    def full(b):
        return (b["bbox"][2] - b["bbox"][0]) > 0.6 * CW or b["label"] == "doc_title"

    items = sorted(blocks, key=lambda b: (b["bbox"][1], b["bbox"][0]))
    bands, cur = [], []
    for b in items:
        if full(b):
            if cur: bands.append(("multi", cur)); cur = []
            bands.append(("full", [b]))
        else:
            cur.append(b)
    if cur: bands.append(("multi", cur))

    def find_columns(bs):
        # merged x-intervals; look for a gutter in the middle 40-60 % with a wide block each side
        iv = sorted((b["bbox"][0], b["bbox"][2]) for b in bs)
        merged = []
        for a, c in iv:
            if merged and a <= merged[-1][1]:
                merged[-1][1] = max(merged[-1][1], c)
            else:
                merged.append([a, c])
        best = None
        for (_, e1), (s2, _) in zip(merged, merged[1:]):
            mid = ((e1 + s2) / 2 - x1) / CW
            if s2 - e1 >= 0.015 * CW and 0.4 <= mid <= 0.6:
                if best is None or (s2 - e1) > (best[1] - best[0]):
                    best = (e1, s2)
        if not best:
            return None
        g = (best[0] + best[1]) / 2
        L = [b for b in bs if center(b["bbox"])[0] < g]; R = [b for b in bs if center(b["bbox"])[0] >= g]
        wide = lambda grp: any((b["bbox"][2] - b["bbox"][0]) >= 0.3 * CW for b in grp)
        return (L, R) if L and R and wide(L) and wide(R) else None

    def by_rows(bs):
        # top-to-bottom, but left-to-right within a row: a fraction's top sits above its '(A)' label's top
        bs = sorted(bs, key=lambda b: (b["bbox"][1], b["bbox"][0]))
        same_row = lambda a, b: a[1] <= center(b)[1] <= a[3] and b[1] <= center(a)[1] <= b[3]
        # ... and an option label beside a picture taller than it ('(A)' left of a drawn structure) comes first
        beside_fig = lambda f, b: f["label"] == "figure" and b["bbox"][2] <= f["bbox"][0] and f["bbox"][1] <= center(b["bbox"])[1] <= f["bbox"][3]
        for i in range(1, len(bs)):
            j = i
            while j > 0 and bs[j]["bbox"][0] < bs[j - 1]["bbox"][0] and (same_row(bs[j - 1]["bbox"], bs[j]["bbox"])
                                                                          or beside_fig(bs[j - 1], bs[j])):
                bs[j - 1], bs[j] = bs[j], bs[j - 1]; j -= 1
        return bs

    out = []
    for kind, bs in bands:
        if kind == "full":
            bs[0]["mode"], bs[0]["col"] = 1, 0; out.append(bs[0]); continue
        cols = find_columns(bs)
        if cols and all(re.match(r"\s*\([A-Da-d1-4]\)", runs_text(b.get("runs") or [])) for b in bs):
            cols = None                             # a 2 x 2 option grid boxed cell by cell: rows, not two columns
        if cols:
            for ci, grp in enumerate(cols):
                for b in by_rows(grp):
                    b["mode"], b["col"] = 2, ci; out.append(b)
        else:
            for b in by_rows(bs):
                b["mode"], b["col"] = 1, 0; out.append(b)
    for i, b in enumerate(out):
        b["order"] = i
    return out


def merge_fragments(blocks, body_h):
    """Re-join paragraphs the layout model split in two (Q6 in Attempt #1)."""
    out = []
    for b in blocks:
        if out and b["label"] == "text" and out[-1]["label"] in ("text", "paragraph_title") and out[-1]["col"] == b["col"] \
                and out[-1]["mode"] == b["mode"] and b["runs"] and out[-1]["runs"]:
            p = out[-1]; ptxt = runs_text(p["runs"]).rstrip(); ctxt = runs_text(b["runs"]).lstrip()
            gap = b["bbox"][1] - p["bbox"][3]
            starts_marker = bool(Q_START_RE.match(ctxt) or re.match(r"\(?[A-Da-d]\)", ctxt) or re.match(r"[•\-–]\s", ctxt))
            lower = bool(re.match(r"[a-z]", ctxt))
            open_end = not re.search(r"[.?!:;]$|\]$", ptxt)
            # a wrapped section title ('1.7 ... and Theorem' / 'of Perpendicular Axes'): one short lowercase line
            title_ok = p["label"] == "text" or (lower and not b["tags"] and b["bbox"][3] - b["bbox"][1] < 2 * body_h)
            if title_ok and not starts_marker and gap < 1.6 * body_h and (lower or (open_end and not re.match(r"[A-Z]", ctxt))):
                p["runs"] = merge_text_runs(p["runs"] + [{"t": "text", "v": " "}] + b["runs"])
                p["bbox"] = union(p["bbox"], b["bbox"])
                for t in b["tags"]:
                    if t not in p["tags"]: p["tags"].append(t)
                p["flags"] += b["flags"] + ["merged_fragment"]
                for k in ("raw", "glyphs"):         # diagnosis keeps both halves' reads
                    if k in b:
                        p[k] = p[k] + ("\n" if k == "raw" else 0) + b[k] if k in p else b[k]
                continue
        out.append(b)
    return out


# --------------------------------------------------------------------------------------
# Figure cropper
# --------------------------------------------------------------------------------------
def crop_figure(img, fig_box, other_boxes, cfg):
    """Ink-based tight crop. Starts from the layout box, grows to touching ink (axis labels, arrows),
    never swallows other layout blocks. Ink = luminance < lum_thresh (default 245 keeps light-grey grids)."""
    import cv2, numpy as np
    H, W = img.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in fig_box]
    E = cfg["fig_expand"]
    rx1, ry1, rx2, ry2 = max(0, x1 - E), max(0, y1 - E), min(W, x2 + E), min(H, y2 + E)
    gray = cv2.cvtColor(img[ry1:ry2, rx1:rx2], cv2.COLOR_BGR2GRAY)
    # a scan's paper is speckled grey (~234): at 245 the specks count as ink and bridge the figure to the print
    # around it (an option's '(C)' was swallowed). Ink must be darker than the specks; clean paper keeps 245.
    light = gray[gray > 200]
    thr = min(cfg["lum_thresh"], float(np.percentile(light, 5)) - 8) if light.size else cfg["lum_thresh"]
    ink = (gray < thr).astype(np.uint8)
    fb = [x1 - rx1, y1 - ry1, x2 - rx1, y2 - ry1]
    for ob in other_boxes:
        if inter(ob, fig_box) / max(1, area(ob)) > 0.5:      # a label that lives inside the figure
            continue
        ox1, oy1, ox2, oy2 = [int(v) for v in (ob[0] - rx1, ob[1] - ry1, ob[2] - rx1, ob[3] - ry1)]
        ox1, oy1 = max(0, ox1), max(0, oy1); ox2, oy2 = min(rx2 - rx1, ox2), min(ry2 - ry1, oy2)
        if ox2 > ox1 and oy2 > oy1:
            # erase the part of the foreign block that lies OUTSIDE the figure box
            for (a, b_, c, d) in [(ox1, oy1, ox2, min(oy2, fb[1])), (ox1, max(oy1, fb[3]), ox2, oy2),
                                  (ox1, oy1, min(ox2, fb[0]), oy2), (max(ox1, fb[2]), oy1, ox2, oy2)]:
                if c > a and d > b_:
                    ink[max(0, b_):d, max(0, a):c] = 0
    k = cfg["fig_join"]
    dil = cv2.dilate(ink, np.ones((k, k), np.uint8))
    n, lab, stats, _ = cv2.connectedComponentsWithStats(dil, connectivity=8)
    keep_ids = [i for i in range(1, n) if inter([stats[i][0], stats[i][1], stats[i][0] + stats[i][2], stats[i][1] + stats[i][3]], fb) > 0]
    flags = []
    if keep_ids:
        mask = np.isin(lab, keep_ids) & (ink > 0)
        ys, xs = np.nonzero(mask)
        bx1, by1, bx2, by2 = xs.min(), ys.min(), xs.max() + 1, ys.max() + 1
    else:
        bx1, by1, bx2, by2 = fb; flags.append("crop_no_ink_used_layout_box")
    pad = cfg["fig_pad"]
    box = [max(0, rx1 + bx1 - pad), max(0, ry1 + by1 - pad), min(W, rx1 + bx2 + pad), min(H, ry1 + by2 + pad)]
    if area(box) > 1.8 * area(fig_box):
        flags.append("crop_much_larger_than_layout_box")
    return box, flags


def glyph_height(gray):
    """Median height (px) of the page's letter-sized ink marks: the yardstick of every size rule (as complete_blocks)."""
    import cv2, numpy as np
    st = cv2.connectedComponentsWithStats((gray < 150).astype(np.uint8), connectivity=8)[2]
    hs = st[1:, 3]; hs = hs[(hs > 8) & (hs < 150)]
    return float(np.median(hs)) if len(hs) else 30.0


def lone_line_below(gray, b, ch):
    """Box of a short print run standing alone at the right end of the line just below b, else None: where an exam
    tag ('[Mar 22]') sits when the question's last line has no room for it. A next question or an option row starts
    at the left, so it never qualifies."""
    import numpy as np
    x1, x2 = max(0, int(b[0])), int(b[2])
    y1, y2 = int(b[3]) + 1, int(min(gray.shape[0], b[3] + 2.6 * ch))
    if y2 - y1 < ch or x2 - x1 < 4 * ch:
        return None
    ink = gray[y1:y2, x1:x2] < 150
    rows = np.nonzero(ink.any(1))[0]
    if not len(rows):
        return None
    r0 = r1 = int(rows[0])
    while r1 + 1 < ink.shape[0] and ink[r1 + 1].any():
        r1 += 1
    cols = np.nonzero(ink[r0:r1 + 1].any(0))[0]
    if not 0.6 * ch <= r1 - r0 <= 1.8 * ch or cols[0] < 0.5 * (x2 - x1) or cols[-1] - cols[0] > 0.45 * (x2 - x1):
        return None
    return [x1 + int(cols[0]) - 8, y1 + r0 - 6, x1 + int(cols[-1]) + 8, y1 + r1 + 6]


def big_type(gray, box, ch):
    """True when the letters inside box are set well above the body text's height (a chapter banner). Measured: chapter
    titles 2.07-3.32 glyph heights; section headings, a boxed 'Chemistry' subject tab, running heads 1.0-1.41."""
    hs = sorted(h for x, y, w, h, _ in _marks(gray, box) if h >= 0.35 * ch)
    return len(hs) >= 3 and hs[len(hs) // 2] >= 1.7 * ch


def _marks(gray, box):
    """Ink marks inside box: [(x, y, w, h, [hole areas])] in box coordinates."""
    import cv2, numpy as np
    x1, y1, x2, y2 = [max(0, int(v)) for v in box]
    ink = (gray[y1:y2, x1:x2] < 150).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)
    cs, hier = cv2.findContours(ink, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    holes = {}
    for k, h in enumerate(hier[0] if hier is not None else []):
        if h[3] >= 0:                               # a hole: credited to the mark its outer contour belongs to
            px, py = cs[h[3]][0][0]
            holes.setdefault(int(lab[py, px]), []).append(cv2.contourArea(cs[k]))
    return [(*[int(v) for v in st[i, :4]], holes.get(i, [])) for i in range(1, n)]


def drawn_structure(gray, box, ch):
    """True when a formula box holds a structure DRAWN in two dimensions, which no linear formula can carry:
    a bond drawn between two rows (the '|' of CH3-CH2-C(-CH2-CH3)), or a charge circle stacked over an atom.
    Measured on 10 pages: fires on all 6 such boxes and on none of 85 others (matrices, fractions, equations)."""
    if box[3] - box[1] < 3 * ch:
        return False                                # one line of maths
    mk = _marks(gray, box)
    ov = lambda x, w, x2, w2, tol: min(x + w, x2 + w2) - max(x, x2) > -tol
    for i, (x, y, w, h, hl) in enumerate(mk):
        if w <= 0.25 * ch and 0.8 * ch <= h <= 2.5 * ch and h >= 5 * w:    # thin and upright: a bond if rows sit at both ends
            above = any(j != i and y2 < y and y - ch <= y2 + h2 <= y + 0.25 * ch and ov(x, w, x2, w2, 0.3 * ch)
                        for j, (x2, y2, w2, h2, _) in enumerate(mk))
            below = any(j != i and y2 + h2 > y + h and y + h - 0.25 * ch <= y2 <= y + h + ch and ov(x, w, x2, w2, 0.3 * ch)
                        for j, (x2, y2, w2, h2, _) in enumerate(mk))
            if above and below:
                return True
        if 0.9 * ch <= max(w, h) <= 2 * ch and abs(w - h) <= 0.1 * max(w, h) and len(hl) >= 2:   # a circled charge (+)
            if any(j != i and ov(x, w, x2, w2, 0.5 * ch) and (0 <= y2 - (y + h) <= 0.8 * ch or 0 <= y - (y2 + h2) <= 0.8 * ch)
                   for j, (x2, y2, w2, h2, _) in enumerate(mk)):
                return True                         # ... stacked over or under an atom, not inline
    return False


def rings_outside(gray, boxed, ch):
    """Boxes of thin-lined closed drawings (a benzene ring) that no layout box covers: the layout model leaves some
    unboxed, and the reader then reads only the 'Mg' beside the ring. Thin strokes (~0.25 glyph height) set them
    apart from big banner digits ('0' of '10', strokes 0.6-1.0); a large hole from letters."""
    import cv2, numpy as np
    ink = (gray < 150).astype(np.uint8)
    cs, hier = cv2.findContours(ink, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    dist = cv2.distanceTransform(ink, cv2.DIST_L2, 3)
    out = []
    for k, h in enumerate(hier[0] if hier is not None else []):
        if h[3] < 0 or cv2.contourArea(cs[k]) < 4 * ch * ch:
            continue                                # a hole of at least 2 x 2 glyph heights
        x, y, w, hh = cv2.boundingRect(cs[h[3]])
        b = [x, y, x + w, y + hh]
        if max(w, hh) > 10 * ch or 2 * float(dist[y:y + hh, x:x + w].max()) > 0.4 * ch:
            continue
        if any(contains_pt(o, center(b), 4) for o in boxed) or b in out:
            continue
        out.append(b)
    return out


def along_row(gray, box, ch):
    """A ring's substituents ('—Mg', '—Mg – Cl') reach past the ring: add the dark, glyph-sized marks on its row
    that sit within a glyph height of it, repeatedly. An option label '(A)' stands ~2.5 glyph heights off: left out."""
    import cv2, numpy as np
    H, W = gray.shape
    wx1, wy1 = max(0, int(box[0] - 12 * ch)), max(0, int(box[1] - ch))
    wx2, wy2 = min(W, int(box[2] + 12 * ch)), min(H, int(box[3] + ch))
    st = cv2.connectedComponentsWithStats((gray[wy1:wy2, wx1:wx2] < 150).astype(np.uint8), connectivity=8)[2][1:]
    marks = [[x + wx1, y + wy1, x + w + wx1, y + h + wy1] for x, y, w, h, _ in st if h <= 2.5 * ch]
    b, grown = [float(v) for v in box], True
    while grown:
        grown = False
        for m in marks:
            if m[0] >= b[0] and m[2] <= b[2] and m[1] >= b[1] and m[3] <= b[3]:
                continue
            if max(m[0] - b[2], b[0] - m[2]) <= ch and min(m[3], b[3]) - max(m[1], b[1]) >= 0.5 * (m[3] - m[1]):
                b, grown = union(b, m), True
    return b


def polish_figure(img, box, ch):
    """A figure crop as it goes into Word -> (BGR image, its box on the page):
    - glyphs the crop's edge cuts through are print around the figure (half the next line, a cut '(C)'): removed;
    - a band or rule across the whole crop, apart from the rest at its top or bottom, is page decoration: trimmed;
    - re-trimmed to the ink with an even margin; paper made white, ink black, specks dropped."""
    import cv2, numpy as np
    H, W = img.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in box]
    if x2 - x1 < 4 or y2 - y1 < 4:
        return img[y1:y2, x1:x2].copy(), [x1, y1, x2, y2]
    m = int(2 * ch)
    X1, Y1, X2, Y2 = max(0, x1 - m), max(0, y1 - m), min(W, x2 + m), min(H, y2 + m)
    dark = (cv2.cvtColor(img[Y1:Y2, X1:X2], cv2.COLOR_BGR2GRAY) < 150).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(dark, connectivity=8)
    ix1, iy1, ix2, iy2 = x1 - X1, y1 - Y1, x2 - X1, y2 - Y1
    cut = [i for i in range(1, n) if st[i, 3] <= 2.5 * ch and st[i, 2] <= 6 * ch          # glyph-sized
           and st[i, 0] < ix2 and st[i, 1] < iy2 and st[i, 0] + st[i, 2] > ix1 and st[i, 1] + st[i, 3] > iy1
           and (st[i, 0] < ix1 or st[i, 1] < iy1 or st[i, 0] + st[i, 2] > ix2 or st[i, 1] + st[i, 3] > iy2)]
    crop = img[y1:y2, x1:x2].copy()
    gone = np.isin(lab[iy1:iy2, ix1:ix2], cut) if cut else np.zeros(crop.shape[:2], bool)
    if cut:                                         # their soft grey rims too, never the print that stays
        keep_ink = (lab[iy1:iy2, ix1:ix2] > 0) & ~gone
        gone = (cv2.dilate(gone.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0) & ~keep_ink
    g = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    g[gone] = 255
    paper = float(np.percentile(g, 90))             # scans print on grey paper: every threshold is relative to it
    # specks, before trimming so that none stretches the box: a small mark with no other print near it (the dot of
    # an 'i' or a radical's dot sit right by their letter)
    mark = (g < paper - 25).astype(np.uint8)
    n, lab2, st, _ = cv2.connectedComponentsWithStats(mark, connectivity=8)
    small = [i for i in range(1, n) if st[i, 4] <= 0.05 * ch * ch and max(st[i, 2], st[i, 3]) <= 0.4 * ch]
    if small:
        r = max(3, int(0.6 * ch))
        near = cv2.dilate(((mark > 0) & ~np.isin(lab2, small)).astype(np.uint8), np.ones((2 * r + 1, 2 * r + 1), np.uint8))
        lone = np.isin(lab2, [i for i in small if not near[lab2 == i].any()])
        gone |= lone; g[lone] = 255
    dark = min(200.0, paper - 40)
    # rows holding print: dark marks, or a light band across the crop (a grey page banner); faint scan grey is neither
    occ = (g < dark).any(1) | ((g < paper - 12).mean(1) >= 0.85)
    segs, s0 = [], None
    for y, v in enumerate(list(occ) + [False]):
        if v and s0 is None:
            s0 = y
        elif not v and s0 is not None:
            segs.append([s0, y]); s0 = None
    def bar(sg):        # a band across the crop (page banner, rule), or a lone thin line at its edge: the answer blank
        if ((g[sg[0]:sg[1]] < paper - 12).mean(1) >= 0.85).any():   # '______' of the line above (a charge's minus
            return True                                                # is far shorter)
        cols = np.nonzero((g[sg[0]:sg[1]] < dark).any(0))[0]
        return sg[1] - sg[0] <= 0.4 * ch and len(cols) > 0 and cols[-1] - cols[0] >= 2 * ch
    while len(segs) > 1 and bar(segs[0]):
        segs.pop(0)
    while len(segs) > 1 and bar(segs[-1]):
        segs.pop()
    if not segs:
        return crop, [x1, y1, x2, y2]
    top, bot = segs[0][0], segs[-1][1]
    ink = g[top:bot] < min(200.0, paper - 40)
    if not ink.any():
        return crop, [x1, y1, x2, y2]
    ys, xs = np.nonzero(ink)
    pad = 8
    a1, a2 = max(0, xs.min() - pad), min(g.shape[1], xs.max() + 1 + pad)
    b1, b2 = max(0, top + ys.min() - pad), min(g.shape[0], top + ys.max() + 1 + pad)
    out, gg = crop[b1:b2, a1:a2].copy(), g[b1:b2, a1:a2]
    out[gone[b1:b2, a1:a2]] = 255
    black = min(float(np.percentile(gg, 1)), 80.0)
    if paper - black > 40:                          # levels: the paper's shade -> white, the darkest ink -> black
        lut = np.clip((np.arange(256) - black) * 255.0 / max(paper - 12 - black, 1), 0, 255).astype(np.uint8)
        out = cv2.LUT(out, lut)
    return out, [int(x1 + a1), int(y1 + b1), int(x1 + a2), int(y1 + b2)]


# --------------------------------------------------------------------------------------
# Model engines
# --------------------------------------------------------------------------------------
def res_dict(r):
    if hasattr(r, "json"):
        d = r.json() if callable(r.json) else r.json
        if isinstance(d, dict):
            return d.get("res", d)
    if isinstance(r, dict):
        return r.get("res", r)
    try:
        return dict(r)
    except Exception:
        return {}


class Engines:
    def __init__(self, cfg, verify=False):
        self.cfg = cfg; self.verify = verify
        self.layout = self.ocr = self.mfd = self.mfr = self.pfr = self.vlm = None
        self.status = {}

    # ---- loading -----------------------------------------------------------------
    def _try(self, name, fn, retries=2):            # 2: one transient download failure must not end the run
        for k in range(retries):
            try:
                t = time.time(); obj = fn(); self.status[name] = f"ok ({time.time() - t:.0f}s)"
                print(f"[load] {name}: {self.status[name]}")
                return obj
            except Exception as ex:
                self.status[name] = f"FAILED: {type(ex).__name__}: {str(ex)[:600]}"
                print(f"[load] {name} attempt {k + 1}: {self.status[name]}")
                if k + 1 < retries:
                    time.sleep(2)
        return None

    def load_all(self):
        c = self.cfg; fast = c["ocr_profile"] == "fast"
        import paddle
        mk = paddle.__version__.startswith("3.2.")     # oneDNN: safe + fast on 3.2.x, crashes on 3.3.0

        def mk_layout():
            if not c["tables"]:
                # Layout boxes are all we use. PPStructureV3 (paddlex 3.3) always loads the 1.4 GB PP-Chart2Table
                # VLM and runs a second OCR pass whose text we discard, so use the bare layout model instead.
                from paddleocr import LayoutDetection
                return construct(LayoutDetection, model_name=c["layout_model"] or "PP-DocLayout_plus-L",
                                 layout_nms=True, device="cpu", enable_mkldnn=mk)
            from paddleocr import PPStructureV3
            kw = dict(use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,
                      use_seal_recognition=False, use_formula_recognition=False, use_chart_recognition=False,
                      use_table_recognition=bool(c["tables"]), use_region_detection=False,
                      text_detection_model_name="PP-OCRv5_mobile_det", text_recognition_model_name="PP-OCRv5_mobile_rec",
                      device="cpu", enable_mkldnn=mk)
            if c["layout_model"]:
                kw["layout_detection_model_name"] = c["layout_model"]
            return construct(PPStructureV3, **kw)

        def mk_ocr():
            from paddleocr import PaddleOCR
            det = "PP-OCRv5_mobile_det" if fast else "PP-OCRv5_server_det"
            rec = "PP-OCRv5_mobile_rec" if fast else "PP-OCRv5_server_rec"
            return construct(PaddleOCR, text_detection_model_name=det, text_recognition_model_name=rec,
                             use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,
                             device="cpu", enable_mkldnn=mk)

        def mk_mfd():
            from ultralytics import YOLO
            if c["mfd_weights"]:
                return YOLO(c["mfd_weights"])
            from huggingface_hub import hf_hub_download, list_repo_files
            errs = []
            for repo in c["mfd_repos"]:
                try:
                    files = list_repo_files(repo)
                    print(f"   [mfd] {repo} files: {files}")
                    cand = [f for f in files if f.endswith((".pt", ".onnx"))]
                    cand.sort(key=lambda f: (not f.endswith(".pt"), f))
                    for f in cand:
                        try:
                            return YOLO(hf_hub_download(repo, f), task="detect")
                        except Exception as ex:
                            errs.append(f"{repo}/{f}: {type(ex).__name__}: {str(ex)[:150]}")
                    if not cand:
                        errs.append(f"{repo}: no .pt/.onnx among {files}")
                except Exception as ex:
                    errs.append(f"{repo}: {type(ex).__name__}: {str(ex)[:150]}")
            raise RuntimeError(" | ".join(errs) + "  -> pass --mfd-weights <path-or-url>")

        def mk_mfr():
            # The HF repos hold only encoder_model.onnx + decoder_model.onnx (optimum export, no KV cache).
            import onnxruntime as ort
            from huggingface_hub import snapshot_download
            from tokenizers import Tokenizer
            prov = [p for p in ("CUDAExecutionProvider", "CPUExecutionProvider") if p in ort.get_available_providers()]
            errs = []
            for repo in c["mfr_models"]:
                try:
                    d = Path(snapshot_download(repo))
                    gen = json.loads((d / "generation_config.json").read_text(encoding="utf-8"))
                    pp = json.loads((d / "preprocessor_config.json").read_text(encoding="utf-8"))
                    self.mfr_name = repo
                    return dict(enc=ort.InferenceSession(str(d / "encoder_model.onnx"), providers=prov),
                                dec=ort.InferenceSession(str(d / "decoder_model.onnx"), providers=prov),
                                tok=Tokenizer.from_file(str(d / "tokenizer.json")),
                                start=gen["decoder_start_token_id"], eos=gen["eos_token_id"], pad=gen.get("pad_token_id", 0),
                                size=(pp["size"]["width"], pp["size"]["height"]),
                                mean=pp["image_mean"], std=pp["image_std"])
                except Exception as ex:
                    errs.append(f"{repo}: {type(ex).__name__}: {str(ex)[:200]}")
            raise RuntimeError(" | ".join(errs))

        def mk_pfr():
            from paddleocr import FormulaRecognition
            return construct(FormulaRecognition, model_name=c["paddle_formula_model"], device="cpu", enable_mkldnn=mk)

        def mk_vlm():
            import torch
            from transformers import AutoProcessor, AutoModelForImageTextToText
            # fp16: the T4 has no bf16; verified stable on PaddleOCR-VL-1.5 (1.8 GB VRAM)
            m = AutoModelForImageTextToText.from_pretrained(c["vlm_model"], dtype=torch.float16).to("cuda").eval()
            return m, AutoProcessor.from_pretrained(c["vlm_model"])

        self.layout = self._try("layout (PP-DocLayout_plus-L)", mk_layout)
        if c["engine"] == "vlm":
            self.vlm = self._try("recogniser (PaddleOCR-VL-1.5, GPU)", mk_vlm)
            if self.vlm is None or self.layout is None:
                raise RuntimeError("model failed to load: " + "; ".join(f"{k}: {v}" for k, v in self.status.items()))
            return
        self.ocr = self._try("text OCR (PaddleOCR)", mk_ocr)
        self.mfd = self._try("formula detector (MFD)", mk_mfd)
        self.mfr = self._try("formula recogniser (MFR)", mk_mfr)
        if self.mfr is None or self.verify:
            self.pfr = self._try("formula recogniser (Paddle fallback/verify)", mk_pfr)

    # ---- inference ---------------------------------------------------------------
    def layout_blocks(self, img_path):
        res = self.layout.predict(str(img_path))
        d = res_dict(res[0]) if res else {}
        out = []
        for b in d.get("parsing_res_list") or []:
            bb = b.get("block_bbox") or b.get("bbox")
            lab = str(b.get("block_label") or b.get("label"))
            out.append(dict(label=lab, bbox=[float(v) for v in bb], html=b.get("block_content") if lab in TABLE_LABELS else None))
        if not out:
            for b in (d.get("layout_det_res") or d).get("boxes", []):        # PPStructureV3 | LayoutDetection
                out.append(dict(label=str(b["label"]), bbox=[float(v) for v in b["coordinate"]], html=None,
                                score=round(float(b["score"]), 3) if b.get("score") is not None else None))
        return out

    def ocr_lines(self, img_bgr):
        res = self.ocr.predict(img_bgr, return_word_box=True)
        d = res_dict(res[0]) if res else {}
        texts = d.get("rec_texts") or []; boxes = d.get("rec_boxes")
        scores = d.get("rec_scores") or [1.0] * len(texts)
        tw = d.get("text_word") or []; twb = d.get("text_word_boxes") or []
        lines = []
        for i, t in enumerate(texts):
            if boxes is not None and len(boxes) > i:
                bb = [float(v) for v in boxes[i]]
            else:
                poly = d["rec_polys"][i]; xs = [p[0] for p in poly]; ys = [p[1] for p in poly]
                bb = [min(xs), min(ys), max(xs), max(ys)]
            words = None                       # word tokens (spaces included) + boxes; used to cut lines at formulas
            if i < len(tw) and i < len(twb) and "".join(tw[i]) == t:
                words = [(w, [float(v) for v in wb]) for w, wb in zip(tw[i], twb[i])]
                ink = [wb for w, wb in words if w.strip()]
                if ink:                        # det boxes are unclipped (padded); spacing logic needs the ink extent
                    bb = [min(b[0] for b in ink), bb[1], max(b[2] for b in ink), bb[3]]
            if t and t.strip():
                lines.append(dict(text=t, bbox=bb, score=float(scores[i]), words=words))
        return trim_overlaps(lines)

    def detect_formulas(self, img_bgr):
        if self.mfd is None:
            return []
        c = self.cfg
        # CPU on purpose: an ONNX model on a CUDA device makes ultralytics demand onnxruntime-gpu
        r = self.mfd.predict(img_bgr, imgsz=c["mfd_imgsz"], conf=c["mfd_conf"], iou=0.45, verbose=False, device="cpu")[0]
        out = []
        for b in r.boxes:
            cls = str(r.names[int(b.cls)]).lower()
            out.append(dict(bbox=[float(v) for v in b.xyxy[0].tolist()], conf=float(b.conf),
                            kind="display" if cls.startswith(("isol", "disp")) else "inline"))
        return out

    def recognise(self, pil_crops):
        """-> list[(latex | None, source)]"""
        res = [(None, "none")] * len(pil_crops)
        if not pil_crops:
            return res
        if self.mfr is not None:
            bs = self.cfg["batch"]
            for i in range(0, len(pil_crops), bs):
                try:
                    for j, t in enumerate(self.mfr_generate(pil_crops[i:i + bs])):
                        res[i + j] = (clean_latex(t), "mfr")
                except Exception as ex:
                    print("   [mfr] batch failed:", str(ex)[:120])
        need = [i for i, r in enumerate(res) if r[0] is None]
        if need and self.pfr is not None:
            import numpy as np
            for i in need:
                try:
                    out = self.pfr.predict(input=np.array(pil_crops[i])[:, :, ::-1], batch_size=1)
                    d = res_dict(out[0]); res[i] = (clean_latex(d.get("rec_formula", "")), "paddle")
                except Exception as ex:
                    print("   [paddle-formula] failed:", str(ex)[:120])
        return res

    def mfr_generate(self, ims, max_len=256):
        """Greedy decode, same as Pix2Text's default generate(): TrOCR preprocessing (bicubic resize, mean/std)."""
        import numpy as np
        from PIL import Image
        m = self.mfr
        x = np.stack([(np.asarray(im.convert("RGB").resize(m["size"], Image.BICUBIC), np.float32) / 255 - m["mean"]) / m["std"]
                      for im in ims]).transpose(0, 3, 1, 2).astype(np.float32)
        hs = m["enc"].run(None, {"pixel_values": x})[0]
        ids = np.full((len(ims), 1), m["start"], np.int64)
        done = np.zeros(len(ims), bool)
        for _ in range(max_len):
            nxt = m["dec"].run(["logits"], {"input_ids": ids, "encoder_hidden_states": hs})[0][:, -1].argmax(-1)
            nxt[done] = m["pad"]            # finished rows: causal decoder, so trailing pads can't affect them
            ids = np.concatenate([ids, nxt[:, None].astype(np.int64)], 1)
            done |= nxt == m["eos"]
            if done.all():
                break
        return [m["tok"].decode(r.tolist(), skip_special_tokens=True) for r in ids[:, 1:]]

    def vlm_read(self, im, prompt, cap=4096):
        """One layout region -> text with inline \\(..\\) / display \\[..\\] LaTeX (prompts per the model card).
        cap = token budget (4096 is PaddleX's own). '' when the read fails; the caller looks again or falls back."""
        import torch
        model, proc = self.vlm
        msgs = [{"role": "user", "content": [{"type": "image", "image": im}, {"type": "text", "text": prompt}]}]
        try:
            inp = proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True,
                                           return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(**inp, max_new_tokens=cap, do_sample=False)
        except Exception as ex:     # one region (CUDA out of memory on an odd crop, ...) must not cost the whole page
            print(f"   [vlm] read failed ({type(ex).__name__}: {str(ex)[:100]})")
            torch.cuda.empty_cache()
            return ""
        gen = out[0][inp["input_ids"].shape[-1]:]
        if len(gen) >= cap:         # the budget is several times what the crop can hold: this is a repetition loop
            print("   [vlm] output hit the token limit (repetition loop)")
            return ""
        return unloop(proc.decode(gen, skip_special_tokens=True).strip(), prompt)

    def recognise_alt(self, pil_crops):
        if self.pfr is None:
            return [None] * len(pil_crops)
        import numpy as np
        out = []
        for im in pil_crops:
            try:
                d = res_dict(self.pfr.predict(input=np.array(im)[:, :, ::-1], batch_size=1)[0])
                out.append(clean_latex(d.get("rec_formula", "")))
            except Exception:
                out.append(None)
        return out


# --------------------------------------------------------------------------------------
# Page processing  ->  IR
# --------------------------------------------------------------------------------------
def is_textish(label):
    return label not in FIG_LABELS | TABLE_LABELS | FORMULA_LABELS | EQNUM_LABELS | DROP_LABELS


def new_block(label, bbox, **kw):
    b = dict(id=None, label=label, bbox=[round(float(v), 1) for v in bbox], runs=[], tags=[], flags=[], mode=1, col=0, order=0)
    b.update(kw); return b


def fallback_blocks_from_lines(lines):
    """Used only if the layout engine is unavailable: group OCR lines into paragraphs."""
    blocks, cur = [], []
    lh = statistics.median([l["bbox"][3] - l["bbox"][1] for l in lines]) if lines else 20
    for l in sorted(lines, key=lambda l: (l["bbox"][1], l["bbox"][0])):
        if cur and (l["bbox"][1] - cur[-1]["bbox"][3] > 0.9 * lh):
            blocks.append(cur); cur = []
        cur.append(l)
    if cur: blocks.append(cur)
    out = []
    for g in blocks:
        bb = g[0]["bbox"]
        for l in g[1:]: bb = union(bb, l["bbox"])
        out.append(dict(label="text", bbox=bb, html=None))
    return out


def trim_overlaps(lines):
    """Det boxes are unclipped, so a hanging question number '1.' can run into the first letter of its stem
    and read '1.1' (seen with the mobile det). Drop that phantom digit when the number box overlaps its
    right-hand neighbour on the same row. (A tighter unclip ratio fixes this but garbles other words.)"""
    for a in lines:
        if re.fullmatch(r"\s*\(?\d{1,3}[.)]\d\s*", a["text"]) and any(
                b is not a and vover(a["bbox"], b["bbox"]) > 0.5 and a["bbox"][0] < b["bbox"][0] < a["bbox"][2] for b in lines):
            a["text"], a["words"] = a["text"].rstrip()[:-1], None
    return lines


def split_line(l, fboxes):
    """An OCR line that spans a masked inline formula is cut at it, so the formula keeps its x-position
    instead of being appended after the whole line ('density and length ... $\\rho$ $L$')."""
    cuts = [center(f)[0] for f in fboxes if contains_pt(l["bbox"], center(f))]
    if not cuts or not l["words"]:
        return [l]
    segs = {}
    for t, wb in l["words"]:
        k = sum(c < (wb[0] + wb[2]) / 2 for c in cuts)
        s = segs.setdefault(k, dict(l, text="", bbox=[1e9, l["bbox"][1], -1e9, l["bbox"][3]]))
        s["text"] += t
        if t.strip():                                  # space tokens would stretch the box into the gap
            s["bbox"][0] = min(s["bbox"][0], wb[0]); s["bbox"][2] = max(s["bbox"][2], wb[2])
    return [s for _, s in sorted(segs.items()) if s["text"].strip()]


def assemble_runs(items, lh):
    """items: dicts with kind text|math, bbox. Returns runs in reading order with sensible spaces."""
    if not items:
        return []
    items = sorted(items, key=lambda it: center(it["bbox"])[1])
    lines, ref = [], None
    for it in items:
        cy = center(it["bbox"])[1]
        if lines and abs(cy - ref) <= 0.55 * lh:
            lines[-1].append(it); ref = statistics.mean(center(x["bbox"])[1] for x in lines[-1])
        else:
            lines.append([it]); ref = cy
    runs = []
    for li, ln in enumerate(lines):
        ln.sort(key=lambda it: it["bbox"][0])
        for k, it in enumerate(ln):
            if k > 0:
                gap = it["bbox"][0] - ln[k - 1]["bbox"][2]
                mixed = it["kind"] != ln[k - 1]["kind"]          # text|math boundary: boxes are tight, gap is small
                if gap > (0.1 if mixed else 0.22) * lh:
                    runs.append({"t": "text", "v": " "})
            elif li > 0:
                prev = runs[-1] if runs else None
                if prev and prev["t"] == "text" and re.search(r"[A-Za-z]-$", prev["v"]) and it["kind"] == "text" and it["text"][:1].islower():
                    prev["v"] = prev["v"][:-1]          # de-hyphenate
                else:
                    runs.append({"t": "text", "v": " "})
            if it["kind"] == "text":
                runs.append({"t": "text", "v": it["text"]})
            else:
                runs.append(dict(t="math", v=None, kind=it["fkind"], img=it["img"], bbox=it["bbox"],
                                 agree=it.get("agree"), _key=it["_key"]))
    return merge_text_runs(runs)


def process_page(eng: Engines, img_path: Path, pno: int, work: Path, cfg, rules):
    import cv2
    from PIL import Image
    img = cv2.imread(str(img_path)); H, W = img.shape[:2]
    crops_dir = work / "crops"; crops_dir.mkdir(exist_ok=True)
    flags_page = []

    print(f"   [p{pno}] layout...")
    # 1) layout -------------------------------------------------------------------
    lblocks = None
    if eng.layout is not None:
        try:
            lblocks = eng.layout_blocks(img_path)
        except Exception as ex:
            flags_page.append(f"layout_failed:{type(ex).__name__}")
            print(f"   [p{pno}] layout failed: {str(ex)[:140]}")
    # 2) formula boxes ------------------------------------------------------------
    fboxes = eng.detect_formulas(img) if eng.mfd is not None else []
    if eng.mfd is None:
        flags_page.append("mfd_unavailable_inline_formulas_not_detected")

    # layout 'formula' blocks <-> MFD IoU agreement
    formulas = []
    lay_formula = [b for b in (lblocks or []) if b["label"] in FORMULA_LABELS]
    used = set()
    for lf in lay_formula:
        hits = [i for i, f in enumerate(fboxes) if iou(lf["bbox"], f["bbox"]) > 0.25 or inter(lf["bbox"], f["bbox"]) / max(1, area(f["bbox"])) > 0.6]
        if hits:
            for i in hits:
                # MFD's embedding/isolated verdict wins: layout also boxes inline option formulas as "formula"
                used.add(i); formulas.append(dict(bbox=fboxes[i]["bbox"], kind=fboxes[i]["kind"], agree="both"))
        else:
            formulas.append(dict(bbox=lf["bbox"], kind="display", agree="layout_only"))
    for i, f in enumerate(fboxes):
        if i not in used:
            formulas.append(dict(bbox=f["bbox"], kind=f["kind"], agree="mfd_only"))

    # 3) figures + tables -----------------------------------------------------------
    textish_boxes = [b["bbox"] for b in (lblocks or []) if is_textish(b["label"])]
    figures, tables = [], []
    for b in (lblocks or []):
        if b["label"] in FIG_LABELS:
            box, fl = crop_figure(img, b["bbox"], textish_boxes, cfg)
            figures.append(dict(bbox=box, flags=fl))
        elif b["label"] in TABLE_LABELS:
            tables.append(b)
    formulas = [f for f in formulas if not any(contains_pt(fg["bbox"], center(f["bbox"])) for fg in figures)
                and not any(contains_pt(t["bbox"], center(f["bbox"])) for t in tables)]

    print(f"   [p{pno}] {len(formulas)} formulas, {len(figures)} figures; OCR...")
    # 4) text OCR on a masked page ---------------------------------------------------
    masked = img.copy()
    for r in [f["bbox"] for f in figures] + [t["bbox"] for t in tables]:
        masked[int(r[1]):int(r[3]), int(r[0]):int(r[2])] = 255
    for f in formulas:
        # 6 px: MFD boxes are tight; with 3 px the OCR read glyph edges as stray chars ('is s', 'L^2 2')
        b = expand(f["bbox"], 6, 6); masked[max(0, int(b[1])):int(b[3]), max(0, int(b[0])):int(b[2])] = 255
    lines = eng.ocr_lines(masked) if eng.ocr is not None else []
    if not lines:
        flags_page.append("no_ocr_text")
    lh = statistics.median([l["bbox"][3] - l["bbox"][1] for l in lines]) if lines else 40.0
    if lblocks is None:
        lblocks = fallback_blocks_from_lines(lines)
        flags_page.append("layout_fallback_line_grouping")
        textish_boxes = [b["bbox"] for b in lblocks]

    # 5) formula recognition ---------------------------------------------------------
    pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for k, f in enumerate(formulas):
        b = f["bbox"]
        cb = [max(0, b[0] - cfg["formula_pad_x"]), max(0, b[1] - cfg["formula_pad_y"]),
              min(W, b[2] + cfg["formula_pad_x"]), min(H, b[3] + cfg["formula_pad_y"])]
        im = pil.crop(tuple(int(v) for v in cb))
        fn = f"p{pno:03d}_f{k:03d}.png"; im.save(crops_dir / fn)
        f["img"] = f"crops/{fn}"; f["_im"] = im
    print(f"   [p{pno}] recognising {len(formulas)} formulas...")
    rec = eng.recognise([f["_im"] for f in formulas])
    alts = eng.recognise_alt([f["_im"] for f in formulas]) if eng.verify else [None] * len(formulas)
    for f, (lx, src), alt in zip(formulas, rec, alts):
        fl = []
        if lx:
            lx = apply_rules(lx, rules, fl)
            lx = sanitize_for_pandoc(lx)
        f.update(latex=lx or None, src=src, flags=fl)
        if alt and lx:
            sim = difflib.SequenceMatcher(None, re.sub(r"\s", "", lx), re.sub(r"\s", "", alt)).ratio()
            if sim < 0.8:
                f["flags"].append(f"engines_disagree({sim:.2f})"); f["alt"] = alt
        if not lx:
            f["flags"].append("recognition_failed")
        del f["_im"]
    # A formula pandoc can't convert is usually text MFD boxed by mistake, or a hallucination on scan noise:
    # read its crop with the text OCR instead of writing junk LaTeX.
    conv = omml_batch([f["latex"] for f in formulas if f["latex"]])
    for f in formulas:
        if (not f["latex"] or conv.get(f["latex"]) is None or textlike(f["latex"])) and eng.ocr is not None:
            crop = cv2.imread(str(work / f["img"]))
            f["text"] = " ".join(l["text"] for l in sorted(eng.ocr_lines(crop), key=lambda l: (l["bbox"][1], l["bbox"][0])))
            f["flags"].append("formula_read_as_text")

    # 6) build blocks -------------------------------------------------------------
    blocks = []
    text_targets = [dict(b, items=[]) for b in lblocks if is_textish(b["label"]) and b["label"] not in DROP_LABELS]
    dropped_boxes = [b for b in lblocks if b["label"] in DROP_LABELS]
    ex = 0.9 * lh

    def find_target(bb, strict_center=True):
        c = center(bb)
        cands = [t for t in text_targets if contains_pt(t["bbox"], c, tol=4)]
        if not cands:
            cands = [t for t in text_targets if contains_pt(expand(t["bbox"], ex, ex * 0.6), c)]
        return min(cands, key=lambda t: area(t["bbox"])) if cands else None

    orphans = []
    fbs = [f["bbox"] for f in formulas]
    for l in (seg for l0 in lines for seg in split_line(l0, fbs)):
        c = center(l["bbox"])
        if any(contains_pt(d["bbox"], c) for d in dropped_boxes):
            continue
        t = find_target(l["bbox"])
        (t["items"] if t else orphans).append(dict(kind="text", text=l["text"], bbox=l["bbox"], _line=l))
    standalone = []
    for k, f in enumerate(formulas):
        key = f"{pno}:{k}"
        it = dict(kind="math", fkind=f["kind"], bbox=f["bbox"], img=f["img"], agree=f["agree"], _key=key)
        t = None if f["kind"] == "display" else find_target(f["bbox"])
        if t is not None:
            t["items"].append(it)
        else:
            standalone.append((it, f))
    fmeta = {f"{pno}:{k}": f for k, f in enumerate(formulas)}

    for t in text_targets:
        if not t["items"]:
            continue
        bb = t["items"][0]["bbox"]
        for it in t["items"][1:]: bb = union(bb, it["bbox"])
        blk = new_block(t["label"], bb, runs=assemble_runs(t["items"], lh))
        blocks.append(blk)
    for l in orphans:
        blocks.append(new_block("text", l["bbox"], runs=[{"t": "text", "v": l["text"]}], flags=["orphan_line"]))
    for it, f in standalone:
        blocks.append(new_block("formula", it["bbox"], runs=assemble_runs([it], lh)))
    for fg in figures:
        blocks.append(new_block("figure", fg["bbox"], flags=list(fg["flags"])))
    for t in tables:
        blocks.append(new_block("table", t["bbox"], html=t.get("html"), flags=["table_formulas_not_recognised"]))
    for b in blocks:                                    # attach recognised latex to the math runs
        for r in b["runs"]:
            if r["t"] == "math":
                f = fmeta[r.pop("_key")]
                r["v"] = f["latex"]; r["src"] = f["src"]; r["flags"] = f["flags"]
                if "text" in f:
                    r.update(t="text", v=f" {f['text']} ")
                if "alt" in f: r["alt"] = f["alt"]
    return finish_page(blocks, img, img_path, pno, crops_dir, lh, dropped_boxes, flags_page)


def is_running_head(b, H):
    """Chapter/page numbers ('01' in the banner, '1' in the footer) and short running heads/feet in the page margins."""
    t = runs_text(b["runs"]).strip(); y1, y2 = b["bbox"][1], b["bbox"][3]
    if b["label"] == "doc_title" or not t or SECTION_RE.fullmatch(t) or Q_START_RE.match(t):
        return False                                # a section title / question near the page edge is content
    if re.fullmatch(r"[\s\-–—|.•()\[\]]*\d{1,4}[\s\-–—|.•()\[\]]*", t) and (y2 < 0.15 * H or y1 > 0.88 * H):
        return True                                 # chapter / page number ('- 12 -'); '= 120' or an equation is content
    if len(t) <= 3 and t.isalnum() and y2 < 0.15 * H and y2 - y1 > 0.03 * H:
        return True                                 # the banner's big '01' read as 'O'; a normal-size 'OR' is content
    # body text starts ~4.7% from the top on these pages, so a short line there ('6.8 Sonometer', 'pipe.') is content
    return (y2 < 0.04 * H or y1 > 0.96 * H) and len(t.split()) <= 4


TAIL_TAG_RE = re.compile(r"(?:\\q?quad|\\[ ,;:]|~|\s)*\[[^\[\]]*\]\s*$")


def finish_page(blocks, img, img_path, pno, crops_dir, lh, dropped_boxes, flags_page):
    """Shared by both engines: text hygiene, tags, running heads, text-like 'formulas', reading order, figure crops."""
    import cv2
    H, W = img.shape[:2]
    for b in blocks:
        runs = []
        for r in b["runs"]:
            tail = TAIL_TAG_RE.search(r["v"]) if r["t"] == "math" and r.get("v") else None
            tag = latex_text(tail.group()) if tail else ""
            if TAG_RE.fullmatch(tag):               # '...=3~[July~22]', '\left[Mar17\right]': the tag was read as maths
                r = dict(r, v=re.sub(r"\\left\s*$", "", r["v"][:tail.start()]).strip())
            if r["t"] == "math" and r.get("v") and (prose := text_only_math(r["v"])):
                runs.append({"t": "text", "v": f" {prose} "}); b["flags"].append("formula_was_text")
            elif r["t"] == "math" and r.get("v") and (textlike(r["v"]) or textlike(unwrap_env(r["v"]))):
                runs += split_textlike(r["v"]); b["flags"].append("formula_was_text")
            elif r["t"] != "math" or r.get("v") or not tail:
                runs.append(r)
            if TAG_RE.fullmatch(tag):
                runs.append({"t": "text", "v": " " + tag})
        for i, r in enumerate(runs):
            if r["t"] != "math" or not r.get("v"):
                continue
            t = latex_text(r["v"])
            if TAG_RE.fullmatch(t) or TAG_RE.fullmatch(f"[{t}]"):      # only an exam tag: split out of a line above
                runs[i] = {"t": "text", "v": f" [{t.strip('[] ')}] "}   # ('\left[\mathrm{Mar} 17\right]'), or bracketless
            elif re.fullmatch(r"(?:\s*\\(?:hline|toprule|midrule|bottomrule|cline\s*\{[^{}]*\}))+\s*", r["v"]):
                runs[i] = {"t": "text", "v": " ______ "}               # the answer blank '______.' read as a table rule
        b["runs"] = merge_text_runs(runs)
        fl = b["flags"]
        for r in b["runs"]:
            if r["t"] == "text":
                r["v"] = clean_text(r["v"], fl)
        b["tags"] = extract_tags(b["runs"])
        for r in b["runs"]:
            if r["t"] == "text":
                r["v"] = re.sub(r"[ \t]{2,}", " ", r["v"])
        if b["runs"] and b["runs"][0]["t"] == "text":
            b["runs"][0]["v"] = b["runs"][0]["v"].lstrip()
        if b["runs"] and b["runs"][-1]["t"] == "text":
            b["runs"][-1]["v"] = b["runs"][-1]["v"].rstrip()
        b["runs"] = [r for r in b["runs"] if not (r["t"] == "text" and not r["v"])]
        if b["label"] not in ("figure", "table") and not b["runs"] and not b["tags"]:
            b["flags"].append("empty")
        elif b["label"] not in ("figure", "table") and is_running_head(b, H):
            b["flags"].append("empty"); dropped_boxes.append(dict(label="running_head", bbox=b["bbox"]))

    blocks = [b for b in blocks if "empty" not in b["flags"]]
    for b in blocks:   # a question/option printed under a figure gets labelled figure_title (or aside_text, ...)
        if b["label"] not in ("text", "figure", "table", "formula", "doc_title", "paragraph_title") and b["runs"] \
                and re.match(r"\s*(?:\(?\d{1,3}\s*[.)](?!\d)|\([A-D]\))", runs_text(b["runs"])):
            b["label"] = "text"; b["flags"].append("relabelled_question")
        elif b["label"] == "text" and all(r["t"] == "text" for r in b["runs"]) and SECTION_RE.fullmatch(runs_text(b["runs"])):
            b["label"] = "paragraph_title"; b["flags"].append("relabelled_section")
        elif b["label"] in ("text", "paragraph_title") and b["bbox"][3] < 0.15 * H and all(r["t"] == "text" for r in b["runs"]) \
                and (m := re.fullmatch(r"\s*(?:\d{1,2}\s+)?([A-Z][A-Za-z ,&'()\-]{2,60}?)\s*", runs_text(b["runs"]))) \
                and big_type(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), b["bbox"], lh / 1.8):
            # the chapter banner the layout model did not box ('02 Solutions'): the title, without its number
            b["label"], b["runs"] = "doc_title", [{"t": "text", "v": m.group(1)}]; b["flags"].append("relabelled_title")
    blocks = [b for b in blocks if b["runs"] or b["label"] in ("figure", "table") or b["tags"]]
    figs = sorted((b for b in blocks if b["label"] == "figure"), key=lambda b: -area(b["bbox"]))
    inside = {id(s) for k, big in enumerate(figs) for s in figs[k + 1:] if inter(big["bbox"], s["bbox"]) >= 0.7 * area(s["bbox"])}
    blocks = [b for b in blocks if id(b) not in inside]    # a crop of part of another crop: the layout boxed it twice
    blocks = order_blocks(blocks)
    blocks = merge_fragments(blocks, lh)
    for i, b in enumerate(blocks):
        b["id"] = f"p{pno}b{i}"; b["order"] = i

    # save figure crops
    for b in blocks:
        if b["label"] == "figure":
            fn = f"p{pno:03d}_fig{b['order']:02d}.png"
            pic, b["bbox"] = polish_figure(img, b["bbox"], lh / 1.8)    # lh ~ 1.8 glyph heights
            cv2.imwrite(str(crops_dir / fn), pic); b["img"] = f"crops/{fn}"
        if b["label"] == "table":
            fn = f"p{pno:03d}_tbl{b['order']:02d}.png"
            x1, y1, x2, y2 = [int(v) for v in b["bbox"]]
            cv2.imwrite(str(crops_dir / fn), img[y1:y2, x1:x2]); b["img"] = f"crops/{fn}"

    return dict(page=pno, w=W, h=H, image=f"pages/{img_path.name}", body_h=round(lh, 1), blocks=blocks,
                dropped=[dict(label=d["label"], bbox=d["bbox"]) for d in dropped_boxes], flags=flags_page)


# --------------------------------------------------------------------------------------
# VLM engine (PaddleOCR-VL-1.5): reads each region's text AND inline math together
# --------------------------------------------------------------------------------------
MATH_RE = re.compile(r"\$\$(.+?)\$\$|\\\[(.+?)\\\]|\\\((.+?)\\\)|\$(.+?)\$", re.S)
_BOOL = r"(?:\\(?:bar|overline)\s*\{[A-Z]\}|[A-Z])"
BOOL_RE = re.compile(rf"\(\s*{_BOOL}(?:\s*(?:\+|\.|\\cdot|\\oplus)\s*{_BOOL})+\s*\)")      # '(A+B)', '(\bar{A} \cdot B)'


def vlm_runs(s):
    """'In \\(4\\) s ... \\[\\frac{a}{b}\\]' -> text/math runs (inline vs display from the delimiter)."""
    runs, pos = [], 0
    for m in MATH_RE.finditer(s):
        if m.start() > pos:
            runs.append({"t": "text", "v": s[pos:m.start()]})
        lx = sanitize_for_pandoc(next(g for g in m.groups() if g is not None))
        bar = re.fullmatch(r"\\(?:over|under)line\s*\{(.*)\}", lx, re.S)
        if bar and balance_braces(bar.group(1)) == bar.group(1) \
                and re.search(r"[(\[]|\\(?:text|mathrm|left)\b|\\ ", bar.group(1)) and not BOOL_RE.fullmatch(bar.group(1)):
            # the '______.' blank above an option row read as a bar over it; a real bar covers bare
            # symbols/digits (\overline{v}, \overline{A+B}, recurring \overline{142857}) or a bracketed Boolean
            # sum (\overline{(A+B)}: dropping that bar turns NOR into OR) and stays
            lx = bar.group(1)
            t = latex_text(lx)
            if TAG_RE.fullmatch(t) or TAG_RE.fullmatch(f"[{t}]"):   # the blank itself, with the tag read on it: keep it
                runs.append({"t": "text", "v": " ______ "})       # (finish_page turns the tag into the question's tag)
        lab = re.match(r"\(([A-D])\)(?![\^_'])\s*(?=\S)", lx)     # '(A)^{-1}' is maths, not a label
        if lab:                     # the option label got pulled inside the math: '(A)\left[L^{2}...\right]'
            runs.append({"t": "text", "v": f"({lab.group(1)}) "}); lx = lx[lab.end():]
        runs.append(dict(t="math", v=lx or None, kind="display" if (m.group(1) or m.group(2)) else "inline",
                         src="vlm", flags=[] if lx else ["recognition_failed"]))
        pos = m.end()
    if pos < len(s):
        runs.append({"t": "text", "v": s[pos:]})
    return runs


def dedupe_boxes(blocks):
    """The layout model emits nested duplicates ('(A) radially inward' inside the box of all four options)."""
    keep = []
    for b in sorted(blocks, key=lambda b: -area(b["bbox"])):
        if all(inter(b["bbox"], k["bbox"]) < 0.6 * area(b["bbox"]) for k in keep):
            keep.append(b)
    return keep


def find_gutter(blocks, W, gray=None):
    """x of the gap between two text columns, or None (single column)."""
    nar = [b["bbox"] for b in blocks if b["label"] == "text" and b["bbox"][2] - b["bbox"][0] < 0.6 * W]
    L = [b[2] for b in nar if center(b)[0] < W / 2]; R = [b[0] for b in nar if center(b)[0] >= W / 2]
    if L and R and max(L) < min(R):
        return (max(L) + min(R)) / 2
    if gray is None:
        return None
    # boxes on one side only (the layout model boxed the other column as a figure, or skipped it): use the
    # whitest x in the middle of the page body. A column rule is ink, but the white beside it is found instead.
    # shortcut: a full-width item crossing the gutter (big figure, wide formula) can hide it; then None as before
    H = gray.shape[0]
    body = gray[int(0.08 * H):int(0.92 * H), int(0.35 * W):int(0.65 * W)] < 150
    low = list(body.sum(0) <= 0.04 * H) + [False]
    runs, s0 = [], None                             # near-white runs of x wider than a word gap
    for x, v in enumerate(low):
        if v and s0 is None:
            s0 = x
        elif not v and s0 is not None:
            if x - s0 >= 0.012 * W:
                runs.append(int(0.35 * W) + (s0 + x) // 2)
            s0 = None
    # nearest the page centre: a hanging-indent channel (question numbers | text) is as white as the gutter
    return min(runs, key=lambda x: abs(x - W / 2)) if runs else None


def complete_blocks(gray, texts, avoid, W, drop=()):
    """Layout models skip small items: option labels '(A)', a hanging '8.', some option formulas. Find dark ink no
    kept box covers; attach it to a text box on the same row (never across the column gutter), else group it into
    one new block per row. Returns (texts, new_blocks, glyph_height); a grown text box is re-read as a whole."""
    import cv2, numpy as np
    ink = (gray < 150).astype(np.uint8)            # dark print only: show-through and grey banners are lighter
    n, _, st, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)
    hs = st[1:, 3]; hs = hs[(hs > 8) & (hs < 150)]
    ch = float(np.median(hs)) if len(hs) else 30.0  # typical glyph height
    # Layout boxes often cut a line in half (a right-aligned '[Oct 08]' on its own line), and the model then
    # skips the half-visible line: grow each box to whole glyphs that are at least half inside it.
    g = st[1:][(st[1:, 3] < 2.5 * ch) & (st[1:, 2] < 6 * ch)].astype(float)
    gx1, gy1, gx2, gy2 = g[:, 0], g[:, 1], g[:, 0] + g[:, 2], g[:, 1] + g[:, 3]
    for t in texts:
        x1, y1, x2, y2 = t["bbox"]
        inside = np.clip(np.minimum(gx2, x2) - np.maximum(gx1, x1), 0, None) * np.clip(np.minimum(gy2, y2) - np.maximum(gy1, y1), 0, None)
        cut = (inside >= 0.5 * g[:, 2] * g[:, 3]) & ((gx1 < x1) | (gx2 > x2) | (gy1 < y1) | (gy2 > y2))
        if cut.any():
            t["bbox"] = [min(x1, gx1[cut].min()), min(y1, gy1[cut].min()), max(x2, gx2[cut].max()), max(y2, gy2[cut].max())]
    for b in [t["bbox"] for t in texts] + avoid:
        x1, y1, x2, y2 = [int(v) for v in expand(b, 4, 4)]
        ink[max(0, y1):y2, max(0, x1):x2] = 0
    dil = cv2.dilate(ink, np.ones((max(3, int(0.4 * ch)), max(3, int(1.2 * ch))), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(dil, connectivity=8)
    comps, strokes = [], []
    for i in range(1, n):
        x, y, w, h = st[i, :4]
        ys, xs = np.nonzero((lab[y:y + h, x:x + w] == i) & (ink[y:y + h, x:x + w] > 0))
        if not len(xs):
            continue
        bb = [x + xs.min(), y + ys.min(), x + xs.max() + 1, y + ys.max() + 1]
        bw, bh = bb[2] - bb[0], bb[3] - bb[1]
        if 2.2 * ch <= bh <= 12 * ch and bw < 0.15 * bh:
            strokes.append([float(v) for v in bb])  # the bracket of a matrix, a tall parenthesis: nothing to read by
            continue                                # itself, but it binds the rows it spans into one region (below)
        if bh < 0.8 * ch or (bh > 4 * ch and bw < 0.15 * bh) or bb[3] < 0.03 * gray.shape[0] or bb[1] > 0.97 * gray.shape[0]:
            continue                                # specks, dashed/solid rules under headings, column rule, page-edge noise
        # (a lone '1', 'l' or 'I' is dense too: only something larger than a glyph is a fill)
        if (len(xs) > 0.5 * bw * bh and (bw > 2 * ch or bh > 2 * ch)) or any(inter(bb, d) > 0 for d in drop) \
                or any(inter(bb, t["bbox"]) > 0.8 * area(t["bbox"]) for t in texts) \
                or (bw > 0.8 * W and (bb[3] < 0.1 * gray.shape[0] or bb[1] > 0.9 * gray.shape[0])):
            continue                                # solid fills, decoration around a running head, a frame/panel
                                                    # behind a kept box, page-wide header/footer bars
        comps.append([float(v) for v in bb])
    g = find_gutter(texts, W, gray)
    side = lambda bb: g is None or center(bb)[0] < g
    orig = [list(t["bbox"]) for t in texts]         # match against the original boxes: growing them would snowball
    rest = []
    tall = strokes + [c for c in comps if c[3] - c[1] >= 2.2 * ch]     # brackets (alone or joined to a column of entries)
    for c in comps:
        if c[3] - c[1] < 2.2 * ch and any(vover(c, t) > 0.5 and max(t[0] - c[2], c[0] - t[2]) < 8 * ch for t in tall):
            # an entry of a bracketed matrix belongs to the matrix, not to the '(B)' printed beside its middle row:
            # attached to the label it was read a second time as '-5 4 (B)'
            rest.append(c); continue
        near = [k for k, ob in enumerate(orig) if ob[1] <= center(c)[1] <= ob[3] and side(c) == side(ob)
                and max(ob[0] - c[2], c[0] - ob[2]) < (10 if c[2] <= ob[0] else 4) * ch   # hanging number/marker
                and c[3] - c[1] <= ob[3] - ob[1]]
        if near:
            t = texts[near[0]]; t["bbox"] = union(t["bbox"], c); t["grown"] = True
        else:
            rest.append(c)
    # one block per visual row (per column); a fraction's numerator line and its label/denominator line are one row.
    # [box, printed]: a bare stroke joins only what stands next to it (the rows of ITS matrix: without it they were
    # read line by line), and a stroke that joined nothing is a rule and is dropped
    rows = [[list(c), True] for c in rest] + [[list(s), False] for s in strokes]
    merged = True
    while merged:
        merged = False
        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                (a, pa), (b, pb) = rows[i], rows[j]
                gap = max(a[1], b[1]) - min(a[3], b[3])
                hov = min(a[2], b[2]) - max(a[0], b[0]) > 0
                beside = (pa and pb) or max(a[0] - b[2], b[0] - a[2]) < 8 * ch
                if side(a) == side(b) and ((vover(a, b) > 0.3 and beside) or (gap < 0.6 * ch and hov)):
                    rows[i] = [union(a, b), pa or pb]; rows.pop(j); merged = True
                    break
            if merged:
                break
    rows = [r for r, printed in rows if printed]
    small = lambda t: t[3] - t[1] <= 2.5 * ch and t[2] - t[0] <= 6 * ch          # an option label '(A)', a number '4.'
    for r in rows:      # a plain text box (mostly) inside a row found here would be read twice: in the row's crop and
        # alone. A label boxed on its own just left of the row ('(A)' before its matrix) belongs to it too: read alone it
        # was placed after a row spanning two option lines, and the options came out empty
        for t in [t for t in texts if t["label"] == "text" and (
                inter(r, t["bbox"]) >= 0.5 * area(t["bbox"])
                or small(t["bbox"]) and side(t["bbox"]) == side(r) and r[1] <= center(t["bbox"])[1] <= r[3]
                and -ch < r[0] - t["bbox"][2] < 6 * ch)]:
            r[:] = union(r, t["bbox"]); texts.remove(t)
    return texts, [dict(label="text", bbox=r, html=None) for r in sorted(rows, key=lambda r: r[1])], ch


def subitem_cuts(gray, b, ch):
    """y of each line (after the first) that opens with a hanging marker ('i.', 'ii.', '(a)') left of the text
    column. The VLM folds 'when, i. the pipe ... ii. the pipe ...' into one sentence and drops the markers; a
    crop that STARTS at a marker keeps it. A marker line: a narrow first ink segment, a gap wider than a word
    space, then text starting where an unmarked line of the region starts. Justified lines ('For   a   conical')
    and option rows ('(A)  3 m/s': no unmarked line starts at their value) don't qualify. Fewer than two -> []."""
    import numpy as np
    x1, y1, x2, y2 = [max(0, int(v)) for v in b]
    ink = gray[y1:y2, x1:x2] < 150
    rows, raw, y = ink.any(1), [], 0
    while y < len(rows):
        if not rows[y]:
            y += 1; continue
        s0 = y
        while y < len(rows) and rows[y]:
            y += 1
        raw.append([s0, y])
    bands = [r for r in raw if r[1] - r[0] >= 0.5 * ch]  # text lines; never merged with each other
    for s0, e0 in (r for r in raw if r[1] - r[0] < 0.5 * ch):   # the dot of an 'i', a bar: to the nearer line
        near = min(bands, key=lambda r: min(abs(s0 - r[1]), abs(r[0] - e0)), default=None)
        if near:
            near[0], near[1] = min(near[0], s0), max(near[1], e0)
    lines = []
    for s0, e0 in bands:
        xs = np.nonzero(ink[s0:e0].any(0))[0]
        if not len(xs):
            continue
        wide = np.nonzero(np.diff(xs) > 1.5 * ch)[0]
        marker = len(wide) and xs[wide[0]] + 1 - xs[0] <= 2.5 * ch
        lines.append(dict(y=s0, start=int(xs[0]), rest=int(xs[wide[0] + 1]) if marker else None))
    plain = [ln["start"] for ln in lines if ln["rest"] is None]
    cuts = [y1 + ln["y"] - 2 for k, ln in enumerate(lines)
            if k and ln["rest"] is not None and any(abs(ln["rest"] - x) < 0.8 * ch for x in plain)]
    return cuts if len(cuts) >= 2 else []


_SEE = r"(?:recogni[sz]e|read|identify|detect|see|make out)"
VLM_FAIL_RE = re.compile(
    r"\W*(?:(?:i'?m |i am )?sorry\b[,.! ]*(?:but )?)?(?:"
    r"(?:the|this) (?:image|picture|text|content)(?: in (?:the|this) (?:image|picture))? (?:is|appears|looks|seems)(?: to be)? "
    r"(?:too |very |quite )?(?:blurry|blurred|unclear|illegible|unreadable)"
    rf"|(?:i (?:am |was )?|i'm )?(?:unable|not able) to {_SEE}"
    rf"|i (?:cannot|can ?not|can't|could ?not|couldn't) {_SEE}"
    r"|(?:there (?:is|are) |there's )?no (?:readable |recogni[sz]able |visible |legible )?(?:text|content)\b)", re.I)
RUNAWAY_RE = re.compile(r"([^_.\-\s=])\1{15,}")


def vlm_failed(txt):
    """The VLM answers unreadable crops with a sentence ('The image is too blurry to recognize ...') or a
    runaway repeat ('1000000...'). Neither is page content. The sentence must OPEN the output: a question that
    says 'a person is unable to read a book ...' or 'the image is too blurry when ...' is content."""
    return bool(VLM_FAIL_RE.match(txt) or RUNAWAY_RE.search(txt))


def pdf_text(cfg, pno, bbox, W):
    """Text the PDF itself carries inside bbox (pixels of the rendered page); '' for image-only PDFs."""
    return " ".join(w for bb, w in page_layer(cfg, pno, W)[1] if contains_pt(bbox, center(bb)))


@functools.lru_cache(maxsize=1)
def _pdf(path):                                     # opened once per run: pdf_text is called per region
    import pymupdf
    return pymupdf.open(path)


def is_text_area(gray, box):
    """True when a box the layout model called a figure/table is really print: hundreds of glyph-sized marks
    carrying most of the ink. Diagrams are dominated by long strokes (axes, bonds, curves) and carry few letters."""
    import cv2, numpy as np
    x1, y1, x2, y2 = [max(0, int(v)) for v in box]
    ink = (gray[y1:y2, x1:x2] < 150).astype(np.uint8)
    n, _, st, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)
    st = st[1:]
    if not len(st):
        return False
    hs = st[:, 3]; ok = hs[(hs > 8) & (hs < 150)]
    ch = float(np.median(ok)) if len(ok) else 30.0
    glyph = (hs >= 0.4 * ch) & (hs <= 2 * ch) & (st[:, 2] <= 3 * ch)
    return int(glyph.sum()) >= 300 and st[glyph, 4].sum() >= 0.6 * st[:, 4].sum()


def split_tall(gray, b, limit=1_000_000):
    """The VLM downscales crops above ~1M px (its max_pixels) and reads small print badly: split an oversized
    region at the widest blank band in its middle half, recursively."""
    x1, y1, x2, y2 = [max(0, int(v)) for v in b]
    h = y2 - y1
    if (x2 - x1) * h <= limit or h < 8:
        return [b]
    blank = (gray[y1:y2, x1:x2] < 150).sum(1) <= 1
    best, y = None, 0
    while y < h:
        if not blank[y]:
            y += 1; continue
        s0 = y
        while y < h and blank[y]:
            y += 1
        if 0.25 * h < (s0 + y) / 2 < 0.75 * h and (best is None or y - s0 > best[1] - best[0]):
            best = (s0, y)
    if best is None:
        return [b]
    cut = y1 + (best[0] + best[1]) // 2
    return split_tall(gray, [b[0], b[1], b[2], cut], limit) + split_tall(gray, [b[0], cut, b[2], b[3]], limit)


@functools.lru_cache(maxsize=1)
def _unlooper():
    try:
        from paddlex.inference.pipelines.paddleocr_vl.uilts import truncate_repetitive_content
        return truncate_repetitive_content
    except Exception as ex:
        print(f"   [vlm] PaddleX's repetition guard is unavailable ({type(ex).__name__}): the built-in checks still apply")


def unloop(s, prompt="OCR:"):
    """A runaway repeat cut back to its content, with PaddleX's own guard and its own thresholds (tables repeat
    legitimately, so only very long ones are touched)."""
    f = _unlooper()
    try:
        return f(s, min_count=5000 if prompt.startswith("Table") else 50) if f else s
    except Exception:
        return s


def otsl_html(s):
    """PaddleOCR-VL answers 'Table Recognition:' in OTSL (<fcel>..<nl>), not HTML; PaddleX converts it the same way."""
    if "<table" in s.lower():
        return s
    try:
        from paddlex.inference.pipelines.paddleocr_vl.uilts import convert_otsl_to_html
        return convert_otsl_to_html(s)
    except Exception as ex:                         # said aloud: a paddlex change would turn every table into text
        print(f"   [table] OTSL conversion failed ({type(ex).__name__}: {str(ex)[:100]}): reading the table as text")
        return ""


# ---- checking a read, and looking again ------------------------------------------------------------------------
# The reader gives no confidence, and its failures are silent: an empty answer for a small crop, a sentence invented
# for a crop with no letters, a region with half its lines missing. Two things it cannot fake are how much is PRINTED
# in a region (counted from the page image) and, when the PDF carries real text, the words themselves.
def glyph_stats(gray, box, ch):
    """(letter-sized marks, height of the tallest mark) printed inside box. White print on a dark banner counts too."""
    import cv2, numpy as np
    x1, y1, x2, y2 = [max(0, int(v)) for v in box]
    ink = gray[y1:y2, x1:x2] < 150
    if not ink.size:
        return 0, 0
    inverted = ink.mean() > 0.5
    if inverted:
        ink = ~ink
    st = cv2.connectedComponentsWithStats(ink.astype(np.uint8), connectivity=8)[2][1:]
    if inverted:        # white letters on a dark banner float inside it; the white margins of a box round a bar touch its edge
        st = st[(st[:, 0] > 0) & (st[:, 1] > 0) & (st[:, 0] + st[:, 2] < ink.shape[1]) & (st[:, 1] + st[:, 3] < ink.shape[0])]
    if not len(st):
        return 0, 0
    w, h = st[:, 2], st[:, 3]
    return int(((h >= 0.35 * ch) & (w <= np.maximum(4 * ch, 2.5 * h))).sum()), int(h.max())


ZERO_WIDTH = (r"(?:frac|dfrac|tfrac|sqrt|left|right|big|Big|bigg|Bigg|mathrm|mathbf|mathit|mathsf|mathtt|boldsymbol|"
              r"operatorname|text|textrm|textbf|textit|overline|underline|vec|hat|bar|dot|ddot|tilde|widehat|"
              r"displaystyle|textstyle|limits|q?quad)")


def print_len(txt):
    """About how many marks the read text prints as: LaTeX markup, blanks ('____') and dot leaders don't count.
    Matrix markup ('\\begin{bmatrix}', '&', '\\\\') prints nothing: counted, it made every matrix read look 1.5-2.5x
    too long, which raised the page's median and hid a read that looped."""
    s = re.sub(r"[_.\-–—…]{3,}", "", txt)
    s = re.sub(r"\\begin\s*\{array\}\s*\{[^{}]*\}|\\(?:begin|end)\s*\{[^{}]*\}|\\\\|&", "", s)
    s = re.sub(rf"\\[()\[\]]|\\(?:{ZERO_WIDTH}(?![A-Za-z])|[,;:! ])", "", s)
    return len(re.sub(r"[\s{}^_\\$]", "", re.sub(r"\\[A-Za-z]+", "x", s)))


def layer_recall(txt, lw):
    """Share of the PDF's own words (lw) that the read text contains."""
    have = re.sub(r"[^a-z0-9]", "", txt.lower())
    return sum(w in have for w in lw) / max(len(lw), 1)


def read_verdict(txt, ng, med, lw=(), sized=True):
    """'' when a read is plausible, else what is wrong with it. ng = letter-sized marks printed in the region,
    med = this page's typical characters-per-mark (it varies with font and scan, so it is measured, not assumed:
    on the reference pages every good read sat within 0.65-1.27 of the median), lw = the PDF's own words there."""
    if not txt.strip() or vlm_failed(txt):
        return "empty"
    if collapse_repeats(txt) != txt:
        return "repeats"                            # the reader looped ('[Mar 18 / A / A / C]' five times): whatever the length
    r = print_len(txt) / max(ng, 1) / med
    if sized and ng >= 20 and not 0.57 <= r <= 1.6:
        return "short" if r < 0.57 else "long"
    if sized and ng < 20 and print_len(txt) > 4 * ng + 20:
        return "long"                               # a sentence where a label or a tag is printed
    if len(lw) >= 6 and layer_recall(txt, lw) < 0.75:
        return "words"
    return ""


def read_fit(txt, ng, med, lw=()):
    """How far a read is from plausible (0 = spot on): keeps the better of two attempts."""
    if not txt.strip() or vlm_failed(txt):
        return float("inf")
    return abs(math.log(max(print_len(txt), 1) / max(ng, 1) / med)) + (2 * (1 - layer_recall(txt, lw)) if len(lw) >= 6 else 0) \
        + (1 if collapse_repeats(txt) != txt else 0)


def collapse_repeats(txt):
    """'X X X X X' -> 'X' when a piece of 20-400 characters holding a letter comes three or more times in a row: the
    reader stuck in a loop. Rows of a zero matrix ('0 & 0 & 0 \\\\') hold no letter and stay."""
    return re.sub(r"(?s)(.{20,400}?)(?:\s*\1){2,}",
                  lambda m: m.group(1) if re.search(r"[A-Za-z]", re.sub(r"\\[A-Za-z]+", "", m.group(1))) else m.group(0), txt)


def on_canvas(im, ch):
    """A crop centred on a page-coloured canvas no smaller than the reader's minimum input: a small crop is
    otherwise magnified many times, and a tight one has no margin."""
    from PIL import Image
    w, h = im.size
    g = max(range(256), key=im.convert("L").histogram().__getitem__)      # the paper's own shade
    out = Image.new("RGB", (max(w + 4 * int(ch), 448), max(h + 4 * int(ch), 252)), (g, g, g))
    out.paste(im, ((out.width - w) // 2, (out.height - h) // 2))
    return out


@functools.lru_cache(maxsize=2)
def _layer(path, pno):
    page = _pdf(path)[pno - 1]
    import pymupdf
    m = page.rotation_matrix                        # words come in unrotated coordinates; the page image is rotated
    words = [(tuple((pymupdf.Rect(w[:4]) * m).normalize()), w[4]) for w in page.get_text("words", sort=not page.rotation)]
    n = hidden = 0
    for sp in page.get_texttrace():                 # type 3 = invisible text: the layer a scanner's OCR adds
        c = len(sp["chars"]); n += c; hidden += c * (sp["type"] == 3 or not sp.get("opacity", 1))
    # a page that is one big picture is a scan, and its text an OCR layer even when it is not marked hidden
    # (NAPS2/PDFsharp write it as ordinary text): never the authority on words or digits
    scan = any(area(i["bbox"]) > 0.7 * area(page.rect) for i in page.get_image_info())
    return ("none" if len(words) < 20 else "ocr" if scan or hidden > 0.5 * n else "real"), words, page.rect.width


def page_layer(cfg, pno, W):
    """(kind, [(bbox in page-image pixels, word)]) for the PDF's own text on a page. kind: 'real' = the text the PDF
    was made from, exact letters and digits; 'ocr' = a hidden layer added by a scanner, a hint at best; 'none'."""
    if not cfg.get("pdf"):
        return "none", []
    kind, words, pw = _layer(cfg["pdf"], pno)
    return kind, [([v * W / pw for v in bb], w) for bb, w in words]


def layer_cores(words, box):
    """Comparable tokens of the PDF's words inside box: words of 3+ letters and numbers of 2+ digits."""
    return [c for bb, w in words if contains_pt(box, center(bb)) for c in re.findall(r"[a-z]{3,}|\d{2,}", w.lower())]


NUM_TOK = r"\d+(?:[.,]\d+)*"


def number_fixes(runs, layer_text):
    """Numbers in the plain-text runs that differ from a real text layer, found by aligning the two word sequences:
    'travels 60 km' read where the PDF says 'travels 50 km'. -> (numbers that agree, [(run index, start, end, new)]).
    Maths runs are left alone: a text layer's rendering of an equation is not reliable."""
    toks = [(ri, m.start(), m.end(), m.group()) for ri, r in enumerate(runs) if r["t"] == "text"
            for m in re.finditer(rf"[A-Za-z]+|{NUM_TOK}", r["v"])]
    b = re.findall(rf"[A-Za-z]+|{NUM_TOK}", layer_text)
    num = lambda s: bool(re.fullmatch(NUM_TOK, s))
    agree, fixes = 0, []
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, [t[3].lower() for t in toks], [t.lower() for t in b],
                                                       autojunk=False).get_opcodes():
        if tag == "equal":
            agree += sum(num(t[3]) for t in toks[i1:i2])
        elif tag == "replace" and i2 - i1 == j2 - j1 <= 3:           # a local disagreement, word for word
            fixes += [(t[0], t[1], t[2], new) for t, new in zip(toks[i1:i2], b[j1:j2])
                      if num(t[3]) and num(new) and abs(len(t[3]) - len(new)) <= 1]
    return agree, fixes


def process_page_vlm(eng: Engines, img_path: Path, pno: int, work: Path, cfg):
    import cv2
    from PIL import Image
    img = cv2.imread(str(img_path)); H, W = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    crops_dir = work / "crops"; crops_dir.mkdir(exist_ok=True)
    print(f"   [p{pno}] layout...")
    lblocks = eng.layout_blocks(img_path)
    dropped, texts, heads = [], [], []
    for b in lblocks:
        if b["label"] in ("header", "footer"):     # '6.8 Sonometer' at the very top of a page gets labelled header
            x1, y1, x2, y2 = [int(v) for v in b["bbox"]]
            t = eng.vlm_read(pil.crop((x1, y1, x2, y2)), "OCR:", 128).strip()
            if SECTION_RE.fullmatch(t):
                texts.append(dict(b, label="paragraph_title")); continue
            if b["label"] == "header" and re.fullmatch(r"[A-Za-z][A-Za-z ,:&'()\-]{2,60}", t) and len(t.split()) <= 6:
                heads.append((b, t))                # a chapter title in the banner? decided once the glyph size is known
        if b["label"] in DROP_LABELS:
            dropped.append(b)
    texts = dedupe_boxes(texts + [b for b in lblocks if is_textish(b["label"]) and b["label"] not in DROP_LABELS])
    formulas = [b for b in lblocks if b["label"] in FORMULA_LABELS]
    ch0 = glyph_height(gray)
    # structures drawn in 2D are pictures: a formula box with a bond drawn between rows or a charge circle over an
    # atom (the reader turned them into arrays of 'I's), and thin-lined rings no layout box covers (it read only 'Mg')
    structs = [f for f in formulas if drawn_structure(gray, f["bbox"], ch0)]
    formulas = [f for f in formulas if f not in structs]
    keep_out = [t["bbox"] for t in texts] + [f["bbox"] for f in formulas]   # print a figure's crop must not grow into
    figures, tables, unread_tables = [], [], []
    for sb in [f["bbox"] for f in structs] + [along_row(gray, r, ch0) for r in rings_outside(gray, [b["bbox"] for b in lblocks], ch0)]:
        box, fl = crop_figure(img, sb, keep_out, cfg)
        figures.append(dict(bbox=box, flags=fl + ["drawn_structure"], layout=sb))
    if figures:
        print(f"   [p{pno}] {len(figures)} drawn structure(s) taken as pictures")
    for b in lblocks:
        lab = b["label"]
        if lab not in FIG_LABELS | TABLE_LABELS:
            continue
        share = area(b["bbox"]) / (W * H)
        if is_text_area(gray, b["bbox"]) and (lab in FIG_LABELS or share > 0.4):
            # the layout model sometimes boxes a whole column/page of questions as one image (or table). Not
            # treating it as one leaves its ink unmasked, so complete_blocks recovers it as text rows below.
            print(f"   [p{pno}] layout '{lab}' box ({100 * share:.0f}% of page) is text: reading it as text")
            continue
        if lab in FIG_LABELS:
            box, fl = crop_figure(img, b["bbox"], keep_out, cfg)
            figures.append(dict(bbox=box, flags=fl, layout=b["bbox"]))
            continue
        x1, y1, x2, y2 = [int(v) for v in b["bbox"]]
        html = otsl_html(eng.vlm_read(pil.crop((x1, y1, x2, y2)), "Table Recognition:"))
        tp = _TableParser(); tp.feed(html)
        if tp.rows:
            tables.append(dict(b, html=html))
        else:                                       # never paste a table as a picture: read it as text instead
            print(f"   [p{pno}] table could not be parsed: reading it as text")
            unread_tables.append(dict(label="text", bbox=b["bbox"], html=None))
    texts = dedupe_boxes(texts + unread_tables)
    # the reader sees the page with figures and tables blanked out: a label row '(A) ... (B)' spans the structure
    # printed between its labels, and the reader turned that ring into text ('(B) Mg - Cl (C')
    from PIL import ImageDraw
    for f in figures:   # the crop as it will be pasted: what it shed (half a label, the line under it) stays readable
        f["bbox"] = polish_figure(img, f["bbox"], ch0)[1]
    pictured = [f["bbox"] for f in figures] + [t["bbox"] for t in tables]
    pil_read = pil.copy(); paint = ImageDraw.Draw(pil_read)
    for bb in pictured:
        paint.rectangle([int(v) for v in bb], fill=(255, 255, 255))
    # formula boxes are not read on their own: their ink is picked up with the row they sit in
    texts, extra, ch = complete_blocks(gray, texts,
                                       [f["bbox"] for f in figures] + [t["bbox"] for t in tables] + [d["bbox"] for d in dropped], W,
                                       drop=[d["bbox"] for d in dropped])
    regions = texts + extra
    print(f"   [p{pno}] reading {len(regions)} regions ({len(extra)} found outside layout boxes)...")
    kind, words = page_layer(cfg, pno, W)
    # a page whose own (real) text holds no exam tag gets no tag re-reads: one reader call per question saved
    tags_possible = kind != "real" or bool(TAG_RE.search(" ".join(w for _, w in words)))
    blocks, flags_page = [], []
    for b, t in heads:                              # banner type ('02 Matrices'): the chapter title, kept; judged by its
        # letters, not its box (a boxed 'Chemistry' tab has a tall box); a running head is set in small type; a
        # publisher's logo prints one big mark, not a word's letters
        n_marks, n_letters = glyph_stats(gray, b["bbox"], ch)[0], len(re.sub(r"\W", "", t))
        if big_type(gray, b["bbox"], ch) and n_marks >= 4 and 0.7 * n_letters <= n_marks <= 1.6 * n_letters:
            blocks.append(new_block("doc_title", b["bbox"], runs=[{"t": "text", "v": t}], raw=t))
            dropped.remove(b)
    cap = lambda ng: min(4096, 256 + 8 * ng)        # token budget from what is printed: a runaway repeat on a
                                                    # one-line crop stops after seconds, not after 4096 tokens
    where = lambda b: f"y={int(b[1])}-{int(b[3])}"

    def first_read(rg, b, lone, ng):
        ys = [b[1]] + ([] if lone else subitem_cuts(gray, b, ch)) + [b[3]]
        parts, flag = [], []
        for k, (ya, yb) in enumerate(zip(ys, ys[1:])):
            # pad only at the region's own edges: across a sub-item or split_tall cut it would reach a neighbour line
            pad_t = 8 if k == 0 and b[1] == rg["bbox"][1] else 0
            pad_b = 8 if k == len(ys) - 2 and b[3] == rg["bbox"][3] else 0
            crop = pil_read.crop((max(0, int(b[0]) - 8), max(0, int(ya) - pad_t), min(W, int(b[2]) + 8), min(H, int(yb) + pad_b)))
            t = eng.vlm_read(crop, "Formula Recognition:" if lone else "OCR:", cap(ng))
            if len(ys) > 2 and (not t.strip() or vlm_failed(t)):
                t = pdf_text(cfg, pno, [b[0], ya, b[2], yb], W); flag = ["vlm_failed_used_pdf_text"]
            parts.append(t.strip())
        return "\n".join(t for t in parts if t), flag

    def reread(b, how, lone, ng):
        """A second look through a different view of the same print (the reader is deterministic: the same crop
        gets the same answer). -> (text, reader calls made)."""
        prompt = "Formula Recognition:" if lone else "OCR:"
        box = lambda q, pad: (max(0, int(q[0]) - pad), max(0, int(q[1]) - pad), min(W, int(q[2]) + pad), min(H, int(q[3]) + pad))
        if how == "canvas":
            return eng.vlm_read(on_canvas(pil_read.crop(box(b, 2)), ch), prompt, cap(ng)).strip(), 1
        halves = split_tall(gray, b, limit=0.6 * area(b))      # smaller pieces are read at a larger scale
        if len(halves) < 2:
            return "", 0
        return "\n".join(t for t in (eng.vlm_read(pil_read.crop(box(q, 4)), prompt, cap(ng)).strip() for q in halves) if t), len(halves)

    # 1) read every region that has print in it
    pieces, blank = [], 0
    for rg, b in ((rg, b) for rg in regions for b in split_tall(gray, rg["bbox"])):
        if any(inter(b, bb) > 0.9 * area(b) for bb in pictured):
            continue                                # print inside a figure is in its picture already
        ng, tallest = glyph_stats(gray, b, ch)
        if tallest < 0.35 * ch or not ng:           # specks, a rule, a banner's edge: the reader invents text ('—') for them
            blank += 1; continue
        lone = any(iou(b, f["bbox"]) > 0.6 for f in formulas)     # a row that is just one formula
        txt, flag = first_read(rg, b, lone, ng)
        pieces.append(dict(rg=rg, b=b, ng=ng, lone=lone, txt=txt, flag=flag,
                           in_figure=any(inter(b, f["layout"]) > 0.9 * area(b) for f in figures)))
    if blank:
        print(f"   [p{pno}] {blank} region(s) hold no print (rules, specks): not read")
    good = [print_len(p["txt"]) / p["ng"] for p in pieces
            if p["ng"] >= 20 and not p["lone"] and not read_verdict(p["txt"], 0, 1, sized=False)]
    med = statistics.median(good) if len(good) >= 5 else 1.05
    if kind == "real":                              # is the layer the text that is printed? (fonts without a proper
        lw = [(c, p["txt"]) for p in pieces if not read_verdict(p["txt"], p["ng"], med, sized=not p["lone"])
              for c in layer_cores(words, p["b"])]                                   # character map give gibberish)
        hit = sum(layer_recall(t, [c]) for c, t in lw)
        if len(lw) >= 30 and hit < 0.6 * len(lw):
            print(f"   [p{pno}] the PDF's own text does not match the print ({100 * hit / len(lw):.0f}% of its words read): not used as a check")
            kind = "ocr"

    # 2) judge each read; look again where it is implausible; never keep silent about what is still wrong
    budget = spent = 8 + len(pieces) // 5           # extra reader calls per page: bounds the time a bad scan can cost
    dead, agree, fixes = 0, 0, []
    for p in pieces:
        rg, b, ng, lone_formula, in_figure, txt, flag = (p[k] for k in ("rg", "b", "ng", "lone", "in_figure", "txt", "flag"))
        lw = layer_cores(words, b) if kind == "real" and not lone_formula else []
        why = read_verdict(txt, ng, med, lw, sized=not lone_formula)
        if why and not in_figure:
            for how in ["canvas"] if lone_formula else ["canvas", "split"] if b[3] - b[1] < 4 * ch else ["split", "canvas"]:
                if budget <= 0:
                    break
                t2, used = reread(b, how, lone_formula, ng)
                budget -= used
                if used and read_fit(t2, ng, med, lw) < read_fit(txt, ng, med, lw):
                    txt, flag = t2, flag + [f"reread_{how}"]
                    why = read_verdict(txt, ng, med, lw, sized=not lone_formula)
                    if not why:
                        break
        if collapse_repeats(txt) != txt:            # still looping after the second looks: one copy, reported below
            txt, flag = collapse_repeats(txt), flag + ["repeats_cut"]
            why = read_verdict(txt, ng, med, lw, sized=not lone_formula) or "repeats"
        if tags_possible and re.match(r"\s*\d{1,3}\s*[.)]", txt) and not TAG_RE.search(txt):
            # the model sometimes drops a right-aligned '[Oct 14]' after a long blank: re-read the last line only
            y = max(b[1], b[3] - 2.2 * ch)
            tail = eng.vlm_read(pil_read.crop((max(0, int(b[0])), max(0, int(y) - 4), min(W, int(b[2]) + 8), min(H, int(b[3]) + 8))), "OCR:", 256)
            txt += "".join(" " + t for t in TAG_RE.findall(tail))
            if not TAG_RE.search(txt):              # tag sits above the option rows, not on the last line
                txt += "".join(" " + t for t in TAG_RE.findall(pdf_text(cfg, pno, b, W)))
            if not TAG_RE.search(txt) and (band := lone_line_below(gray, b, ch)):    # ... alone on the line below
                txt += "".join(" " + t for t in TAG_RE.findall(eng.vlm_read(pil_read.crop(tuple(band)), "OCR:", 64)))
        if not txt.strip() or vlm_failed(txt):
            txt = pdf_text(cfg, pno, b, W)
            if not txt and in_figure:
                continue                            # a diagram's own label: it is in the picture already
            if not txt:
                dead += 1
                if ng >= 20:                        # a line or more of real print: its picture beats a hole in the text
                    print(f"   [p{pno}] region at {where(b)} could not be read: pasted as a picture")
                    flags_page.append("region_as_picture")
                    blocks.append(new_block("figure", b, flags=["unread_as_picture"]))
                else:
                    print(f"   [p{pno}] small region at {where(b)} could not be read: left out")
                    flags_page.append("region_unreadable")
                continue
            flag = ["vlm_failed_used_pdf_text"]
        elif why == "words" and layer_recall(txt, lw) < 0.5:
            print(f"   [p{pno}] region at {where(b)}: the read misses most of the PDF's own words, using those (equations there become plain text)")
            txt, flag = pdf_text(cfg, pno, b, W), ["vlm_poor_used_pdf_text"]
            flags_page.append("region_doubtful")
        elif why and not in_figure:
            print(f"   [p{pno}] doubtful read at {where(b)} ({why}): kept, check it")
            flags_page.append("region_doubtful"); flag = flag + [f"doubtful_{why}"]
        if in_figure and len(txt.strip()) <= 15:
            continue                                # a diagram's own label ('mg', 'θ'): it is in the picture already
        runs = vlm_runs(txt)
        if lone_formula and all(r["t"] == "text" for r in runs) and txt.strip():
            runs = vlm_runs(f"\\[{txt}\\]")                              # formula prompt sometimes omits delimiters
        aw = [c for c in lw if c.isalpha()]
        if len(aw) >= 5 and layer_recall(txt, aw) >= 0.9:               # layer and read agree on the words: compare numbers
            a, fx = number_fixes(runs, pdf_text(cfg, pno, b, W))
            agree += a; fixes += [(runs, f) for f in fx]
        label = "formula" if lone_formula and len(runs) == 1 and runs[0]["t"] == "math" else rg["label"]
        blocks.append(new_block(label, b, runs=runs, flags=flag + (["outside_layout"] if rg in extra else []),
                                raw=txt, glyphs=ng, **({"first_read": p["txt"]} if p["txt"] != txt else {})))    # for diagnosis
    if spent - budget:
        print(f"   [p{pno}] {spent - budget} extra read(s) spent on doubtful regions" + (" (page limit reached)" if budget <= 0 else ""))
    if len(pieces) >= 6 and dead > len(pieces) / 2:
        raise RuntimeError(f"the reader returned nothing for {dead} of {len(pieces)} regions")
    if fixes and len(fixes) <= max(2, 0.1 * (agree + len(fixes))):      # a few slips among many agreeing numbers
        for runs, (ri, s, e, new) in sorted(fixes, key=lambda f: (id(f[0]), f[1][0], -f[1][1])):   # right to left
            print(f"   [p{pno}] number set right from the PDF's own text: {runs[ri]['v'][s:e]} -> {new}")
            runs[ri]["v"] = runs[ri]["v"][:s] + new + runs[ri]["v"][e:]
    elif fixes:                                     # they differ everywhere: no telling which is right
        print(f"   [p{pno}] {len(fixes)} numbers differ between the read and the PDF's own text: nothing changed, check them")
        flags_page.append("numbers_differ")
    for fg in figures:
        blocks.append(new_block("figure", fg["bbox"], flags=list(fg["flags"])))
    for t in tables:
        blocks.append(new_block("table", t["bbox"], html=t["html"]))
    ir = finish_page(blocks, img, img_path, pno, crops_dir, 1.8 * ch, dropped, flags_page)    # line height ~ 1.8 glyphs
    # for diagnosis only: what the layout model boxed and which regions were read (blocks carry the reader's raw text)
    ir["layout"] = [dict(label=b["label"], bbox=[round(v) for v in b["bbox"]], score=b.get("score")) for b in lblocks]
    ir["regions"] = [dict(bbox=[round(v) for v in p["b"]], glyphs=p["ng"], source="outside_layout" if p["rg"] in extra else p["rg"]["label"])
                     for p in pieces]
    ir["text_layer"], ir["glyph_h"], ir["chars_per_glyph"] = kind, round(ch, 1), round(med, 2)
    return ir


# --------------------------------------------------------------------------------------
# docx builder (native OMML through pandoc)
# --------------------------------------------------------------------------------------
def omml_batch(latexes):
    """Convert many LaTeX strings with ONE pandoc call. Returns {latex: lxml <m:oMath> | None}."""
    import pypandoc
    from lxml import etree
    import tempfile
    uniq = [l for l in dict.fromkeys(latexes) if l]
    result = {l: None for l in uniq}
    if not uniq:
        return result

    def convert(group):
        md = "\n\n".join("$" + l + "$" for l in group)
        with tempfile.TemporaryDirectory() as td:
            out = os.path.join(td, "m.docx")
            pypandoc.convert_text(md, "docx", format="markdown", outputfile=out, extra_args=["--wrap=none"])
            with zipfile.ZipFile(out) as z:
                root = etree.fromstring(z.read("word/document.xml"))
        paras = root.findall(".//w:body/w:p", NS)
        if len(paras) != len(group):
            return None
        res = {}
        for l, p in zip(group, paras):
            om = p.findall("m:oMath", NS)
            res[l] = om[0] if len(om) == 1 else None
        return res

    try:
        r = convert(uniq)
    except Exception as ex:
        print("   [pandoc] batch failed:", str(ex)[:120]); r = None
    if r is None:                                       # mismatch -> per-formula
        r = {}
        for l in uniq:
            try:
                r.update(convert([l]) or {l: None})
            except Exception:
                r[l] = None
    result.update(r)
    return result


class _TableParser(HTMLParser):
    def __init__(self):
        super().__init__(); self.rows = []; self._cell = None; self._row = None
    def handle_starttag(self, tag, attrs):
        a = dict(attrs)
        if tag == "tr": self._row = []
        elif tag in ("td", "th"):
            self._cell = dict(text="", cs=int(a.get("colspan", 1) or 1), rs=int(a.get("rowspan", 1) or 1))
    def handle_endtag(self, tag):
        if tag in ("td", "th") and self._cell is not None and self._row is not None:
            self._row.append(self._cell); self._cell = None
        elif tag == "tr" and self._row is not None:
            self.rows.append(self._row); self._row = None
    def handle_data(self, data):
        if self._cell is not None: self._cell["text"] += data


class Builder:
    def __init__(self, cfg, omml):
        from docx import Document
        from docx.shared import Pt, Inches
        self.D = Document(); self.cfg = cfg; self.omml = omml
        self.Pt, self.Inches = Pt, Inches
        sec = self.D.sections[0]
        sec.page_width, sec.page_height = Inches(8.27), Inches(11.69)
        for m in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
            setattr(sec, m, Inches(0.8))
        self.text_w = 8.27 - 1.6
        st = self.D.styles["Normal"]; st.font.name = "Cambria"; st.font.size = Pt(11)
        self.stats = dict(native=0, text=0, math_total=0)
        self.work = None
        self.first_title = True
        self.pic_max = None                         # widest an inline picture may be (set while laying out options)
        self.chem = False                           # a chemistry document: table cells get its formula rules too

    def col_width(self):
        return self.text_w           # spec 3.1.7: source columns are linearised, Word output is single-column

    # -- math
    def add_math(self, p, run, where):
        self.stats["math_total"] += 1
        el = self.omml.get(run.get("v")) if run.get("v") else None
        if el is not None:
            p._p.append(copy.deepcopy(el)); self.stats["native"] += 1; return
        # pandoc couldn't build an equation: write it as editable text, never as a picture of text
        p.add_run(" " + latex_text(run.get("v") or "") + " "); self.stats["text"] += 1

    def fill(self, p, runs, where):
        for r in runs:
            if r["t"] == "text":
                if r["v"]: p.add_run(r["v"].replace("\n", " "))
            elif r["t"] == "img":                   # a structure that is an option: inline, at print size or its column's
                if r.get("img") and (self.work / r["img"]).exists():
                    from PIL import Image
                    w = Image.open(self.work / r["img"]).size[0]
                    p.add_run().add_picture(str(self.work / r["img"]),
                                            width=self.Inches(min(w / self.cfg["dpi"], self.pic_max or self.col_width())))
            else:
                self.add_math(p, r, where)

    def tag(self, p, tags):
        from docx.enum.text import WD_TAB_ALIGNMENT
        if tags:
            p.paragraph_format.tab_stops.add_tab_stop(self.Inches(self.col_width()), WD_TAB_ALIGNMENT.RIGHT)
            r = p.add_run("\t" + " ".join(tags)); r.font.size = self.Pt(10)

    # -- structure helpers
    def para(self, style=None, left=0.0, first=0.0, align=None, after=4, keep=False):
        p = self.D.add_paragraph(style=style)
        pf = p.paragraph_format
        pf.space_after = self.Pt(after); pf.space_before = self.Pt(0)
        if left: pf.left_indent = self.Inches(left)
        if first: pf.first_line_indent = self.Inches(first)
        if align is not None: p.alignment = align
        if keep: pf.keep_with_next = True
        return p

    def picture(self, path, max_w=None):
        from PIL import Image
        from docx.enum.text import WD_ALIGN_PARAGRAPH
        w, h = Image.open(path).size
        wi = min(w / self.cfg["dpi"], max_w or self.col_width(), 9.0 * w / h)    # 9 in tall at most: one page
        p = self.para(align=WD_ALIGN_PARAGRAPH.CENTER, after=6)
        p.add_run().add_picture(str(path), width=self.Inches(wi))

    def table(self, b):
        tp = _TableParser(); tp.feed(b.get("html") or "")
        rows = tp.rows
        if not rows:
            if b.get("img"): self.picture(self.work / b["img"])
            return
        ncols = max(sum(c["cs"] for c in r) for r in rows)
        t = self.D.add_table(rows=len(rows), cols=ncols); t.style = "Table Grid"
        occupied = set()
        for ri, r in enumerate(rows):
            ci = 0
            for c in r:
                while (ri, ci) in occupied: ci += 1
                if ci >= ncols: break
                cell = t.cell(ri, ci); self.fill(cell.paragraphs[0], strip_runs(cell_runs(c["text"], self.chem)), "table")
                ce, re_ = min(ncols - 1, ci + c["cs"] - 1), min(len(rows) - 1, ri + c["rs"] - 1)
                for rr in range(ri, re_ + 1):
                    for cc in range(ci, ce + 1):
                        occupied.add((rr, cc))
                if (ce, re_) != (ci, ri):
                    try: cell.merge(t.cell(re_, ce))
                    except Exception: pass
                ci = ce + 1
        self.D.add_paragraph().paragraph_format.space_after = self.Pt(2)

    def display(self, run, where):
        from docx.enum.text import WD_ALIGN_PARAGRAPH
        p = self.para(align=WD_ALIGN_PARAGRAPH.CENTER, after=6)
        el = self.omml.get(run.get("v")) if run.get("v") else None
        if el is None:
            return self.add_math(p, run, where)
        from lxml import etree
        etree.SubElement(p._p, f"{{{OMML_NS}}}oMathPara").append(copy.deepcopy(el))
        self.stats["native"] += 1; self.stats["math_total"] += 1

    def options(self, opts, where, style="A"):
        """Uniform option layout: 4 across, 2 x 2 or one per line, chosen from the widest option so that tab stops
        never overflow (the source's own arrangement is lost once columns are linearised)."""
        labs = sorted(opts)
        if not labs:
            return
        w = max(vis_len(opts[l]) for l in labs)
        per = 4 if w <= 14 else 2 if w <= 34 else 1
        cell = (self.text_w - 0.4) / per
        for i in range(0, len(labs), per):
            row = labs[i:i + per]
            if per == 1:
                p = self.para(left=0.8, first=-0.4, after=2, keep=i + per < len(labs))
                p.paragraph_format.tab_stops.add_tab_stop(self.Inches(0.8))
            else:
                p = self.para(after=2, keep=i + per < len(labs))
                for j in range(per):
                    p.paragraph_format.tab_stops.add_tab_stop(self.Inches(0.4 + j * cell))
                    p.paragraph_format.tab_stops.add_tab_stop(self.Inches(0.4 + j * cell + 0.35))
            self.pic_max = (self.text_w - 1.2) if per == 1 else cell - 0.45
            for l in row:
                p.add_run(("" if per == 1 else "\t") + f"({LABELS[style]['ABCD'.index(l)]})\t")   # as printed: (a) / (1)
                self.fill(p, strip_runs(opts[l]), where)
            self.pic_max = None
        p.paragraph_format.space_after = self.Pt(6)

    def subitems(self, p, items, tail, where):
        """i. / ii. lines hang under the stem; 'Hence derive ...' after them returns to the stem indent."""
        for lab, rs in items:
            p = self.para(left=0.8, first=-0.4, after=2)
            p.paragraph_format.tab_stops.add_tab_stop(self.Inches(0.8))
            p.add_run(f"{lab}\t"); self.fill(p, strip_runs(rs), where)
        if tail:
            p = self.para(left=0.4); self.fill(p, strip_runs(tail), where)
        return p

    def question(self, q, where):
        head, items, tail = split_subitems(q["stem"])
        p = self.para(left=0.4, first=-0.4, keep=bool(q["opts"] or q["figs"] or items))
        p.paragraph_format.tab_stops.add_tab_stop(self.Inches(0.4))
        if q["num"] is not None:
            p.add_run(f"{q['num']}.\t").bold = True
        self.fill(p, strip_runs(head), where)
        self.tag(self.subitems(p, items, tail, where), q["tags"])
        for f in q["figs"]:                     # figures and tables printed between the stem and its options
            if f["label"] == "table":
                self.table(f)
            elif f.get("img"):
                self.picture(self.work / f["img"])
        self.options(q["opts"], where, q.get("style", "A"))

    # -- main
    def add_items(self, items):
        for it in items:
            k, where = it["kind"], it.get("id", "")
            if k == "q":
                self.question(it, where)
            elif k == "figure":
                if it["block"].get("img"): self.picture(self.work / it["block"]["img"])
            elif k == "table":
                self.table(it["block"])
            elif k == "or":
                from docx.enum.text import WD_ALIGN_PARAGRAPH
                self.para(align=WD_ALIGN_PARAGRAPH.CENTER).add_run("OR").bold = True
            elif k == "heading":
                if it["label"] == "doc_title":
                    p = self.D.add_paragraph(style="Title" if self.first_title else "Heading 1"); self.first_title = False
                elif re.match(r"\s*\d{1,2}\s*\.\s*\d{1,2}\s", runs_text(it["runs"])):
                    p = self.D.add_paragraph(style="Heading 2")    # '1.7 ...': box height varies with wrapping
                else:
                    p = self.D.add_paragraph(style="Heading 1" if it["h"] >= 1.25 * it["bh"] else "Heading 2")
                self.fill(p, strip_runs(it["runs"]), where); self.tag(p, it["tags"])
            else:                                   # paragraph (a lone display formula is centred)
                head, items, tail = split_subitems(it["runs"])
                runs = strip_runs(head)
                if len(runs) == 1 and runs[0]["t"] == "math" and runs[0].get("kind") == "display" and not items:
                    self.display(runs[0], where); continue
                if runs or items or it["tags"]:
                    p = self.para(); self.fill(p, runs, where)
                    self.tag(self.subitems(p, items, tail, where), it["tags"])


SUB_RE = re.compile(r"(?:^|(?<=\s))\(?(i{1,3}|iv|vi{0,3}|ix|x)[.)](?=\s)")
ROMANS = ["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"]
TAIL_RE = re.compile(r"(?<=[.?;])\s+(?=(?:Hence|Thus|Therefore|Also|Further|Then|Find|Calculate|Derive|Deduce|Show|"
                     r"State|Explain|Draw|Write|Obtain|Prove|Compare|Which|Choose|Select|Identify|Name)\b)")


def split_subitems(runs):
    """'... a pipe, when, i. the pipe is open ii. the pipe is closed. Hence derive ...' ->
    (head, [('i.', runs), ('ii.', runs)], tail). Only numerals counting up from 'i' qualify, so 'i.e.' or a
    lone 'v.' (velocity) stay text. The tail is what follows the last item from a 'Hence/Also/Find ...' sentence."""
    marks = []
    for ri, r in enumerate(runs):
        if r["t"] != "text":
            continue
        for m in SUB_RE.finditer(r["v"]):
            if m.start() == 0 and ri > 0:           # right after an equation: not a list marker
                continue
            marks.append((ri, m.start(), m.end(), m.group(1), m.group(0).strip()))
    seq = []
    for mk in marks:
        if len(seq) < len(ROMANS) and mk[3] == ROMANS[len(seq)]:
            seq.append(mk)
        elif mk[3] == "i" and len(seq) < 2:
            seq = [mk]
    if len(seq) < 2:
        return runs, [], []
    head, items, target = [], [], None
    target = head
    for ri, r in enumerate(runs):
        if r["t"] != "text":
            target.append(r); continue
        pos = 0
        for m in (m for m in seq if m[0] == ri):
            if r["v"][pos:m[1]]:
                target.append({"t": "text", "v": r["v"][pos:m[1]]})
            items.append([m[4], []]); target = items[-1][1]; pos = m[2]
        if r["v"][pos:]:
            target.append({"t": "text", "v": r["v"][pos:]})
    tail, last = [], items[-1][1]
    for k, r in enumerate(last):
        m = TAIL_RE.search(r["v"]) if r["t"] == "text" else None
        if m:
            tail = [{"t": "text", "v": r["v"][m.end():]}] + last[k + 1:]
            items[-1][1] = last[:k] + [{"t": "text", "v": r["v"][:m.start()]}]
            break
    return head, [tuple(i) for i in items], tail


def strip_runs(runs):
    rs = [dict(x) for x in runs]
    for x in rs:
        if x["t"] == "text":
            x["v"] = re.sub(r"\s+", " ", x["v"])
    if rs and rs[0]["t"] == "text": rs[0]["v"] = rs[0]["v"].lstrip()
    if rs and rs[-1]["t"] == "text": rs[-1]["v"] = rs[-1]["v"].rstrip()
    return [x for x in rs if not (x["t"] == "text" and not x["v"])]


def vis_len(runs):
    """Approximate printed width in characters (a fraction is as wide as its longer part)."""
    n = 0
    for r in runs:
        if r["t"] == "text":
            n += len(r["v"].strip()); continue
        if r["t"] == "img":                         # ~12 characters to the inch of printed width
            n += round(r.get("w", 0) / CFG["dpi"] * 12); continue
        s = r.get("v") or ""
        for _ in range(5):
            s = re.sub(r"\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}", lambda m: max(m.group(1), m.group(2), key=len), s)
        n += len(re.sub(r"\\[A-Za-z]+|[{}^_ ]", "", s)) + s.count("\\sqrt") + 1
    return n


# question number at the start of a block or line; option label at the start or after a space/bracket.
# A label is only a CANDIDATE: '(A)' also appears in 'Both (A) and (B)', 'Assertion (A):', 'P (A)', and '(1)'
# in 'equations (1) and (2)'. choose_options() decides, per question, which candidates are its options.
EV_RE = re.compile(r"(?P<or>(?:^|(?<=\n))[ \t]*OR[ \t]*(?=\n|$)"            # 'OR' alone on a line: alternative follows,
                   r"|(?<=[.?!:;\]])[ \t]+OR[ \t]*$)"                          # or read onto the end of the line above it
                   r"|(?P<q>(?:^|(?<=\n))[ \t]*\(?(?P<n>\d{1,3})[ \t]*[.)](?!\d)[ \t]*)"
                   r"|(?P<o>(?:^|(?<=[\s)\]]))\((?P<l>[A-Da-d1-4])\)[ \t]*)")
LABELS = {"A": "ABCD", "a": "abcd", "1": "1234"}     # option label styles, in printed order


def block_events(runs):
    """-> [(kind, val)]: ('run', run) | ('q', {n, raw, bstart}) | ('opt', {style, idx, raw, bstart, lstart, qn})."""
    ev = []
    for i, r in enumerate(runs):
        if r["t"] != "text":
            ev.append(("run", r)); continue
        v, pos = r["v"], 0
        for m in EV_RE.finditer(v):
            if m.group("q") and m.start() == 0 and i > 0:      # '^' marks a question only at the block start
                continue
            if m.start() > pos:
                ev.append(("run", {"t": "text", "v": v[pos:m.start()]}))
            if m.group("or"):
                ev.append(("or", None)); pos = m.end(); continue
            raw, before = m.group(0), v[:m.start()]
            bstart = i == 0 and not before.strip()
            lstart = bstart or before.rstrip(" \t").endswith("\n")
            num = re.fullmatch(r"[ \t]*\(([1-4])\)[ \t]*", raw) if m.group("q") else None   # '(2)' = option or question
            if m.group("q") and not num:
                ev.append(("q", dict(n=int(m.group("n")), raw=raw, bstart=bstart)))
            else:
                l = num.group(1) if num else m.group("l")
                style = "1" if l.isdigit() else "A" if l.isupper() else "a"
                ev.append(("opt", dict(style=style, idx=LABELS[style].index(l), raw=raw, bstart=bstart,
                                       lstart=lstart, qn=int(l) if num else None)))
            pos = m.end()
        if pos < len(v):
            ev.append(("run", {"t": "text", "v": v[pos:]}))
    while ev and ev[0][0] == "run" and ev[0][1]["t"] == "text" and not ev[0][1]["v"].strip():
        ev.pop(0)
    return ev


def choose_options(cands):
    """Which label candidates of one question are its options: the longest run of ONE style in printed order
    ((A)(B)(C)(D), a label skipped only when it appears nowhere later). Ties go to the later run, so in
    'Assertion (A): ... (A) Both A and R ...' the second '(A)' starts the options. A label behind the run
    ('(D) Both (A) and (B)') or of another style stays text. A run must start at its first label or at a line
    start, and numbered labels only at a line start ('using equations (1) and (2)' is not an option list)."""
    def run(k):
        st, acc, nxt = cands[k]["style"], [k], cands[k]["idx"] + 1
        for j in range(k + 1, len(cands)):
            c = cands[j]
            if c["style"] == st and (c["idx"] == nxt or (c["idx"] > nxt and not any(
                    d["style"] == st and d["idx"] == nxt for d in cands[j + 1:]))):
                acc.append(j); nxt = c["idx"] + 1
        return acc
    best = None
    for k, c in enumerate(cands):
        if not c["lstart"] and (c["style"] == "1" or c["idx"] > 0):
            continue
        acc = run(k)
        if len(acc) < 2 and not c["lstart"]:
            continue
        key = (len(acc), c["idx"] == 0, k)
        if best is None or key > best[0]:
            best = (key, acc)
    return {cands[j]["pos"] for j in best[1]} if best else set()


def picture_option(q, fig):
    """Which option a picture printed among a question's options is: the first one whose label came with nothing
    after it, or, when the crop took the label in with the structure, the next label on that row. None: the
    picture follows a finished question and stands on its own."""
    empty = [k for k in "ABCD" if k in q["opts"] and not strip_runs(q["opts"][k])]
    if empty:
        return empty[0]
    nxt = next((k for k in "ABCD" if k not in q["opts"] and all(j not in q["opts"] for j in "ABCD"["ABCD".index(k) + 1:])), None)
    return nxt if nxt and q.get("_opt_box") and vover(fig["bbox"], q["_opt_box"]) > 0.3 else None


def to_items(blocks):
    """Re-segment the reading-order stream into questions/options/paragraphs. Layout blocks cut a question
    anywhere (a label in one box, its formula in another, the stem continuing in the next column); the printed
    markers '12.' and '(A)'..'(D)' are what actually define the structure."""
    items, q, target = [], None, None
    evs = [None if b["label"] in ("doc_title", "paragraph_title", "figure", "table") else block_events(b["runs"])
           for b in blocks]

    def close():
        nonlocal q
        if q is not None:
            items.append(q); q = None

    def new_q(num, bi, ei, id_):
        """A question owns the events up to the next question. Read ahead to that point once: numbered lines
        inside it are statements ('1. Torque is a vector.'), and choose_options picks its option labels."""
        cands, sub, prev, rows, last = [], set(), "", False, 0
        for bj in range(bi, len(blocks)):
            if evs[bj] is None:                         # heading, figure, table
                if blocks[bj]["label"] == "figure" or (blocks[bj]["label"] == "table" and not rows):
                    continue                            # a diagram before the options, or a structure that IS an option
                break
            for ej in range(ei if bj == bi else 0, len(evs[bj])):
                kind, val = evs[bj][ej]
                if kind == "or":
                    break                               # the alternative question is not part of this one
                if kind == "run":
                    if val["t"] != "text" or val["v"].strip():
                        prev = val["v"] if val["t"] == "text" else ""     # an equation after 'Solve:' ends the colon
                    continue
                if kind == "opt" and val["qn"] is not None and val["bstart"] and num is not None and val["qn"] == num + 1:
                    break                               # '(2) A body ...': next question, numbered in brackets
                if kind == "q":
                    # '1.' '2.' statements inside this question: they count up from 1, come before any option row,
                    # and sit inside a block (or open one right after a ':'). A block opening with the next
                    # number is the next question.
                    nxt = num is not None and val["n"] == num + 1
                    stmt = num is not None and not rows and val["n"] == last + 1 and (
                        (not val["bstart"] and (last > 0 or not nxt))
                        or (val["bstart"] and not nxt and (last > 0 or prev.rstrip().endswith(":"))))
                    if not stmt:
                        break                           # the next question
                    sub.add((bj, ej)); last = val["n"]; continue
                cands.append(dict(val, pos=(bj, ej))); rows |= val["lstart"]
            else:
                continue
            break
        return dict(kind="q", num=num, stem=[], opts={}, tags=[], figs=[], id=id_, style="A",
                    _opts=choose_options(cands), _sub=sub)

    complete = lambda q: set("ABCD") <= set(q["opts"])
    for bi, b in enumerate(blocks):
        lab = b["label"]
        if lab != "figure" and lab != "table" and runs_text(b["runs"]).strip() == "OR":
            close(); items.append(dict(kind="or")); continue    # a centred 'OR' (sometimes labelled a title)
        if lab in ("doc_title", "paragraph_title"):
            close(); items.append(dict(kind="heading", label=lab, runs=b["runs"], tags=b["tags"], id=b.get("id", ""),
                                       h=b["bbox"][3] - b["bbox"][1], bh=b.get("_bh", 40))); continue
        if lab in ("figure", "table"):
            slot = picture_option(q, b) if q is not None and q["opts"] and lab == "figure" else None
            if q is not None and not q["opts"]:
                q["figs"].append(b)                 # diagram between a stem and its options
            elif slot:                              # a structure printed as the option itself: '(C) [benzene ring]'
                if slot not in q["opts"]:
                    q["opts"][slot] = target = []
                q["opts"][slot].append(dict(t="img", img=b.get("img"), w=b["bbox"][2] - b["bbox"][0]))
            else:
                close(); items.append(dict(kind=lab, block=b))
            continue
        ev = evs[bi]
        if ev and ev[0][0] not in ("q", "opt", "or"):
            if q is not None and (complete(q) or (lab != "formula" and not q["opts"]
                                                  and re.search(r"[.?!:;]\s*$", runs_text(q["stem"])))):
                close()
            if q is None:
                para = dict(kind="para", runs=[], tags=[], id=b.get("id", "")); items.append(para); target = para["runs"]
            else:
                target.append({"t": "text", "v": " "})
        first_q = alt = None
        trailing_or = False
        start_q, block_qs = q, []                   # for handing out this block's tags (see below)
        for ei, (kind, val) in enumerate(ev):
            if kind == "or":                        # 'OR' inside a block: close, the rest is the alternative
                if ei == len(ev) - 1:               # '... [Mar 18] OR' ending the block: the alternative is the next
                    trailing_or = True; continue    # block; close after this block's tags went to their owner
                close(); items.append(dict(kind="or"))
                alt = dict(kind="para", runs=[], tags=[], id=b.get("id", "")); items.append(alt); target = alt["runs"]
                block_qs.append((ei, alt))          # an alternative ends with its own tag, like a question
                continue
            if kind == "opt" and val["qn"] is not None and val["bstart"] and (q is None or val["qn"] == (q["num"] or 0) + 1) \
                    and (q is None or (bi, ei) not in q["_opts"]):
                kind, val = "q", dict(val, n=val["qn"])    # '(2) A body ...' numbers a question
            if kind == "q":
                if q is not None and (bi, ei) in q["_sub"]:
                    gap = "" if target and target[-1]["t"] == "text" and target[-1]["v"][-1:].isspace() else " "
                    target.append({"t": "text", "v": gap + val["raw"].strip() + " "})    # a statement, not a question
                    continue
                close(); q = new_q(val["n"], bi, ei + 1, b.get("id", ""))
                target = q["stem"]; first_q = first_q or q; block_qs.append((ei, q))
            elif kind == "opt":
                if q is None:                       # options whose question number was lost
                    q = new_q(None, bi, ei, b.get("id", ""))
                key = "ABCD"[val["idx"]]
                if (bi, ei) in q["_opts"]:
                    q["opts"][key] = target = []; q["style"] = val["style"]; q["_opt_box"] = b["bbox"]
                elif val["bstart"] and key in q["opts"]:
                    target = []                     # the same label read twice (layout boxed it twice): drop the repeat
                else:
                    target.append({"t": "text", "v": val["raw"]})   # '(D) Both (A) and (B)', 'Assertion (A):'
            else:
                target.append(val)
        owner = first_q or q or (items[-1] if items and items[-1]["kind"] == "para" else None)
        if owner is None and b["tags"]:
            owner = dict(kind="para", runs=[], tags=[], id=b.get("id", "")); items.append(owner)
        tags = b["tags"]
        # questions and OR alternatives this block holds text of, in order: the one it continues (if its text comes
        # before the first new number or OR) and those it starts. Tags print at each one's end, so with one tag each
        # they pair up in order ('5. ... [July 16]  6. ... [July 17]', '1. ... [Mar 10] OR ... [July 16] OR ...');
        # otherwise an alternative keeps the last tag and the rest go to the first.
        holders = ([start_q] if start_q is not None and (not block_qs or block_qs[0][0] > 0) else []) + [qq for _, qq in block_qs]
        if len(holders) >= 2 and len(tags) == len(holders):
            for h, t in zip(holders, tags):
                if t not in h["tags"]:
                    h["tags"].append(t)
            tags = []
        elif alt is not None and owner is not alt and len(tags) >= 2:
            alt["tags"] += tags[-1:]; tags = tags[:-1]  # each alternative ends with its own tag; positions are lost
        for t in tags if owner is not None else ():
            if t not in owner["tags"]:
                owner["tags"].append(t)
        if trailing_or:
            close(); items.append(dict(kind="or"))
    close()
    return items


def build_docx(pages, out_path: Path, work: Path, cfg):
    # chemistry is settled here, at build time, so a --rebuild of stored pages gets it too
    chem = is_chem_doc([b for pg in pages for b in pg["blocks"]])
    blocks = [dict(b, _bh=pg.get("body_h", 40), runs=chemify(b["runs"], chem)) for pg in pages for b in pg["blocks"]]
    latexes = [r["v"] for b in blocks for r in b["runs"] if r["t"] == "math" and r.get("v")]
    for b in blocks:                                # equations inside table cells
        tp = _TableParser(); tp.feed(b.get("html") or "")
        latexes += [r["v"] for row in tp.rows for c in row for r in cell_runs(c["text"], chem) if r["t"] == "math" and r.get("v")]
    B = Builder(cfg, omml_batch(latexes)); B.work = work; B.chem = chem
    B.add_items(to_items(blocks))
    B.D.save(str(out_path))
    return B.stats


# --------------------------------------------------------------------------------------
# CLI
# --------------------------------------------------------------------------------------
def gpu_vlm_available():
    try:
        import torch, transformers
        return torch.cuda.is_available() and int(transformers.__version__.split(".")[0]) >= 5
    except Exception:
        return False


def parse_pages(s, n):
    if not s: return list(range(1, n + 1))
    out = []
    for part in s.split(","):
        a, _, b = part.partition("-"); out += list(range(int(a), int(b or a) + 1))
    return [p for p in out if 1 <= p <= n]


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("pdf", nargs="?"); ap.add_argument("-o", "--out", default="output.docx")
    ap.add_argument("--work", default="work"); ap.add_argument("--cache", default=None, help="model cache dir (e.g. on Google Drive)")
    ap.add_argument("--subject", default="physics", choices=list(SUBJECT_RULES), help="symbol repair rules (hybrid engine only)")
    ap.add_argument("--rules", help="json: [[regex, replacement, tag], ...] extra LaTeX repair rules (hybrid engine only)")
    ap.add_argument("--pages"); ap.add_argument("--force", action="store_true"); ap.add_argument("--rebuild", action="store_true")
    ap.add_argument("--accurate", action="store_true", help="server OCR models (much slower on CPU)"); ap.add_argument("--verify", action="store_true", help="second formula engine; flags disagreements")
    ap.add_argument("--engine", default="auto", choices=["auto", "vlm", "hybrid"],
                    help="vlm = PaddleOCR-VL-1.5 on a CUDA GPU (best); hybrid = CPU OCR + formula models; auto picks")
    ap.add_argument("--check-env", action="store_true")
    ap.add_argument("--tables", action="store_true", help="table HTML via PP-StructureV3 (also loads its 1.4 GB chart model; much slower)")
    ap.add_argument("--mfd-weights", help="path/URL to formula-detector weights (.pt/.onnx)")
    a = ap.parse_args(argv)

    configure_env(a.cache)
    if a.check_env:
        sys.exit(0 if check_env() else 1)
    if not a.pdf and not a.rebuild:
        ap.error("pdf required")
    work = Path(a.work); (work / "pages").mkdir(parents=True, exist_ok=True); (work / "ir").mkdir(exist_ok=True)
    cfg = dict(CFG); cfg["subject"] = a.subject
    if a.accurate: cfg["ocr_profile"] = "accurate"
    cfg["tables"] = a.tables; cfg["mfd_weights"] = a.mfd_weights; cfg["engine"] = a.engine; cfg["pdf"] = a.pdf
    rules = list(SUBJECT_RULES[a.subject]) + ([tuple(r) for r in json.loads(Path(a.rules).read_text(encoding="utf-8"))] if a.rules else [])
    t0 = time.time()

    if not a.rebuild:
        import pymupdf as fitz
        doc = fitz.open(a.pdf); n = len(doc)
        src = dict(pdf=str(Path(a.pdf).resolve()), size=os.path.getsize(a.pdf), mtime=os.path.getmtime(a.pdf), dpi=cfg["dpi"])
        stamp = work / "source.json"
        if stamp.exists() and json.loads(stamp.read_text(encoding="utf-8")) != src:
            print("[work] different PDF than last run: clearing cached pages/IR")
            for f in list((work / "pages").glob("*.png")) + list((work / "ir").glob("*.json")):
                f.unlink()
        stamp.write_text(json.dumps(src), encoding="utf-8")
        todo = parse_pages(a.pages, n)
        for pno in todo:
            fp = work / "pages" / f"page_{pno:03d}.png"
            if not fp.exists():
                doc[pno - 1].get_pixmap(dpi=cfg["dpi"], alpha=False).save(str(fp))
        need = [p for p in todo if a.force or not (work / "ir" / f"page_{p:03d}.json").exists()]
        if need:
            if cfg["engine"] == "auto":
                cfg["engine"] = "vlm" if gpu_vlm_available() else "hybrid"
            print(f"[engine] {cfg['engine']}")
            eng = Engines(cfg, verify=a.verify); eng.load_all()
            for pno in need:
                t = time.time()
                try:
                    pg = work / "pages" / f"page_{pno:03d}.png"
                    read = lambda: (process_page_vlm(eng, pg, pno, work, cfg) if cfg["engine"] == "vlm"
                                    else process_page(eng, pg, pno, work, cfg, rules))
                    try:
                        ir = read()
                    except Exception:               # a GPU hiccup or memory spike passes: one more go before giving up
                        traceback.print_exc()
                        print(f"[page {pno}] failed; trying it once more")
                        try:
                            import torch; torch.cuda.empty_cache()
                        except Exception:
                            pass
                        ir = read()
                except Exception as ex:
                    traceback.print_exc()
                    ir = dict(page=pno, w=0, h=0, image=f"pages/page_{pno:03d}.png", body_h=40, dropped=[],
                              flags=[f"PAGE_FAILED:{type(ex).__name__}:{str(ex)[:100]}"], blocks=[])
                    import cv2
                    im = cv2.imread(str(work / "pages" / f"page_{pno:03d}.png")); h, w = im.shape[:2]
                    ir.update(w=w, h=h)
                    raw = doc[pno - 1].get_text("text").strip()     # lines kept: questions split at '12.' line starts
                    if raw:
                        print(f"[page {pno}] FAILED -> using the PDF's own text layer for this page (no equations)")
                        ir["blocks"] = [new_block("text", [0, 0, w, h], id=f"p{pno}b0", runs=[{"t": "text", "v": raw}],
                                                  flags=["page_failed_pdf_text"])]
                    else:
                        print(f"[page {pno}] FAILED and the PDF has no text layer -> page inserted as a picture")
                        ir["blocks"] = [new_block("figure", [0, 0, w, h], id=f"p{pno}b0", img=f"pages/page_{pno:03d}.png", flags=["whole_page_image_fallback"])]
                irp = work / "ir" / f"page_{pno:03d}.json"      # atomic: a run killed mid-write must not leave half an IR
                irp.with_suffix(".tmp").write_text(json.dumps(ir, ensure_ascii=False, indent=1), encoding="utf-8")
                os.replace(irp.with_suffix(".tmp"), irp)
                print(f"[page {pno}] {len(ir['blocks'])} blocks in {time.time() - t:.0f}s  {ir['flags']}")
            (work / "engine_status.json").write_text(json.dumps(eng.status, indent=1), encoding="utf-8")
    irs = sorted((work / "ir").glob("page_*.json"))
    pages = [json.loads(p.read_text(encoding="utf-8")) for p in irs]
    if a.pages and not a.rebuild:
        keep = set(parse_pages(a.pages, 10 ** 6)); pages = [p for p in pages if p["page"] in keep]
    if not pages:
        sys.exit("no IR pages found")

    st = build_docx(pages, Path(a.out), work, cfg)
    as_text = f", {st['text']} written as text" if st["text"] else ""
    print(f"\n{a.out}  | {len(pages)} page(s), {st['native']}/{st['math_total']} equations native{as_text}"
          f"  | {time.time() - t0:.0f}s")
    notes = []                                      # everything that did not go cleanly, by page: nothing stays silent
    for p in pages:
        f = p["flags"]
        what = ["could not be read, its plain text or a picture of the page was used"] if any(x.startswith("PAGE_FAILED") for x in f) else []
        what += [f"{f.count(k)} {label}" for k, label in (("region_as_picture", "unread region(s) pasted as a picture"),
                                                          ("region_unreadable", "small unread region(s) left out"),
                                                          ("region_doubtful", "doubtful read(s)")) if k in f]
        what += ["numbers differ from the PDF's own text"] if "numbers_differ" in f else []
        if what:
            notes.append(f"page {p['page']}: {', '.join(what)}")
    print("CHECK against the PDF -> " + "; ".join(notes) if notes else
          "Nothing was left out, pasted as a picture or flagged as doubtful.")


if __name__ == "__main__":
    main()


## Convert

In [ ]:
import os, subprocess
DOCS, FAILED, NOTES = [], [], []
for pdf in PDFS:
    stem = os.path.splitext(pdf)[0]
    out = stem + ".docx"
    # one work dir per file: pages already read are kept, so running this cell again after an interruption
    # resumes at the first unfinished page (the script itself starts clean when the PDF changed)
    work = os.path.join("work", stem)
    print(f"\n===== {pdf}", flush=True)
    p = subprocess.Popen([sys.executable, "-u", "pdf2word_hybrid.py", pdf, "-o", out, "--work", work],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace")
    for line in p.stdout:
        print(line, end="", flush=True)
        if line.startswith("CHECK against the PDF"):   # the converter's own list of pages that did not read cleanly
            NOTES.append(f"{out}: {line.strip()}")
    if p.wait() == 0 and os.path.exists(out):
        DOCS.append(out)
    else:
        FAILED.append(pdf)
if FAILED:
    print("\nFAILED:", ", ".join(FAILED), " (see the log above)")
if not DOCS:
    raise RuntimeError("Nothing was converted; see the log above.")

## Download

In [ ]:
import zipfile
from google.colab import files
if len(DOCS) == 1:
    files.download(DOCS[0])
else:                                                   # browsers block a burst of downloads: send one zip
    with zipfile.ZipFile("converted_word_files.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for d in DOCS:
            z.write(d)
    files.download("converted_word_files.zip")
print("Done:", ", ".join(DOCS))
print("\n".join(NOTES) if NOTES else "No page was flagged: nothing was left out, pasted as a picture or marked doubtful.")
if FAILED:
    print("NOT converted:", ", ".join(FAILED))

## If a result looks wrong (optional)
Set `DIAGNOSTICS = True` in the cell below and run that cell on its own while the runtime is still connected. It downloads `diagnostics.zip`: the PDFs, the Word files and a record of what the converter saw on every page (layout boxes, each region and what was read in it). To report a problem (for example in a GitHub issue), attach that file with a note on which page and question are wrong. It contains your PDFs, so share it only if you are allowed to.

In [ ]:
DIAGNOSTICS = False   # True, then run this cell: downloads what the converter saw (Run all leaves this off)
if DIAGNOSTICS:
    import glob, zipfile
    from google.colab import files
    with zipfile.ZipFile("diagnostics.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for f in PDFS + DOCS + sorted(glob.glob("work/*/ir/*.json")) + sorted(glob.glob("work/*/*.json")):
            z.write(f)
    files.download("diagnostics.zip")